# DeepMzyme Training Notebook

This notebook runs DeepMzyme training through the existing command-line interface in `src/train.py`. It supports two runtime modes:

- `local`: Ubuntu / PyCharm / Jupyter from a local checkout.
- `colab`: Google Colab with Google Drive, Colab dependency setup, and `/content` runtime paths.

The notebook does not duplicate training logic. It builds a CLI command, runs it only when you explicitly enable training, then calls `src/report_runs.py` to create a comparison CSV and optional figure.


In [83]:
#@title Runtime mode and base paths
from pathlib import Path
import os
import shutil
import sys

def _running_in_colab():
    if os.environ.get("COLAB_RELEASE_TAG") or os.environ.get("COLAB_GPU"):
        return True
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

RUN_MODE = "auto"  #@param ["auto", "local", "colab"]
if RUN_MODE == "auto":
    RUN_MODE = "colab" if _running_in_colab() else "local"
elif RUN_MODE not in {"local", "colab"}:
    raise ValueError("RUN_MODE must be 'auto', 'local', or 'colab'.")

# Local defaults are intended for Ubuntu + PyCharm/Jupyter from the repository root.
LOCAL_REPO_ROOT = "/home/mechti/PycharmProjects/DeepMzyme"  #@param {type:"string"}
LOCAL_DATA_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data"  #@param {type:"string"}
LOCAL_OUTPUT_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/notebook_outputs"  #@param {type:"string"}
LOCAL_BUNDLE_PATH = ""  #@param {type:"string"}
LOCAL_UNPACK_DIR = "/home/mechti/PycharmProjects/DeepMzyme/.notebook_bundle"  #@param {type:"string"}
LOCAL_EMBEDDINGS_DIR = os.environ.get("DEEPMZYME_ESM_EMBEDDINGS_DIR", "/media/Data/DeepMzyme_Data/esm_embeddings")  #@param {type:"string"}
LOCAL_RING_EXE_PATH = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/ring-4.0/out/bin/ring"  #@param {type:"string"}
LOCAL_RING_EDGE_OUTPUT_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/RING_features"  #@param {type:"string"}
LOCAL_PRECOMPUTED_RING_EDGES_DIR = "/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/RING_features"  #@param {type:"string"}

# Colab data source options: upload_file, huggingface_link, or drive.
COLAB_DATA_SOURCE = "huggingface_link"  #@param ["upload_file", "huggingface_link", "drive"]
if COLAB_DATA_SOURCE not in {"upload_file", "huggingface_link", "drive"}:
    raise ValueError("COLAB_DATA_SOURCE must be 'upload_file', 'huggingface_link', or 'drive'.")
COLAB_HUGGINGFACE_BUNDLE_URL = "https://huggingface.co/datasets/GMBioinformatics/DeepMzyme/resolve/main/DeepMzyme_Data_runtime_local_2026-05-03.tar.zst"  #@param {type:"string"}
COLAB_HUGGINGFACE_BUNDLE_SHA256 = "c86faa40ff69c021de02b72b5fef9ebd1712f5ef8e6cb3da27b3a9e8261816c1"  #@param {type:"string"}

# Colab defaults keep Drive for persistent data and /content for runtime-local work.
COLAB_REPO_ROOT = "/content/DeepMzyme"  #@param {type:"string"}
COLAB_DRIVE_DATA_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data"  #@param {type:"string"}
COLAB_LOCAL_DATA_DIR = "/content/deepmzyme_data"  #@param {type:"string"}
COLAB_OUTPUT_DIR = "/content/deepmzyme_outputs"  #@param {type:"string"}
COLAB_UNPACK_DIR = "/content/deepmzyme_bundle"  #@param {type:"string"}
COLAB_BUNDLE_PATH = ""  #@param {type:"string"}
COLAB_EMBEDDINGS_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/esm_embeddings"  #@param {type:"string"}
COLAB_LOCAL_EMBEDDINGS_DIR = "/content/deepmzyme_data/esm_embeddings"  #@param {type:"string"}
COLAB_RING_EXE_PATH = "DeepMzyme_Data/ring-4.0/out/bin/ring"  #@param {type:"string"}
COLAB_RING_EDGE_OUTPUT_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/RING_features"  #@param {type:"string"}
COLAB_LOCAL_RING_EDGE_OUTPUT_DIR = "/content/deepmzyme_data/RING_features"  #@param {type:"string"}
COLAB_PRECOMPUTED_RING_EDGES_DIR = "/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/RING_features"  #@param {type:"string"}
COLAB_LOCAL_PRECOMPUTED_RING_EDGES_DIR = "/content/deepmzyme_data/RING_features"  #@param {type:"string"}

REPO_GIT_URL = "https://github.com/MECHTI1/DeepMzyme.git"  #@param {type:"string"}
REPO_GIT_BRANCH = "main"  #@param {type:"string"}
DATASET_NAME = "train_and_test_sets_structures_non_overlapped_pinmymetal"  #@param {type:"string"}
BUNDLE_FILENAME = "DeepMzyme_Data_runtime_local_2026-05-03.tar.zst"  #@param {type:"string"}
DRIVE_OUTPUT_SUBDIR = "Train_Parameters_Models_and_Results"  #@param {type:"string"}

print("RUN_MODE:", RUN_MODE)
print("Current working directory:", Path.cwd())
print("Python executable:", sys.executable)
print("Expected local DeepMzyme interpreter:", "/home/mechti/miniconda3/envs/DeepMzyme/bin/python")
if RUN_MODE == "local" and Path(sys.executable).resolve() != Path("/home/mechti/miniconda3/envs/DeepMzyme/bin/python").resolve():
    print("Note: local mode uses the active notebook kernel. Switch the Jupyter/PyCharm kernel if you need the DeepMzyme conda environment.")



RUN_MODE: local
Current working directory: /home/mechti/PycharmProjects/DeepMzyme/notebooks
Python executable: /home/mechti/miniconda3/envs/DeepMzyme/bin/python
Expected local DeepMzyme interpreter: /home/mechti/miniconda3/envs/DeepMzyme/bin/python


## 1. Optional Google Drive Mount

Drive mounting is opt-in because Colab Drive authentication can block when the notebook is launched through PyCharm's Google Colab integration. Leave `MOUNT_DRIVE=False` for runtime-local `/content` paths; set it to `True` only when the Drive auth prompt is available.


In [84]:
#@title Optional Google Drive mount
MOUNT_DRIVE = RUN_MODE == "colab" and COLAB_DATA_SOURCE == "drive"  #@param {type:"boolean"}
DRIVE_MOUNT_TIMEOUT_MS = 30000  #@param {type:"integer"}

if RUN_MODE == "colab":
    if MOUNT_DRIVE:
        if COLAB_DATA_SOURCE != "drive":
            print('Warning: MOUNT_DRIVE=True but COLAB_DATA_SOURCE is not drive; mounting anyway.')
        from google.colab import drive
        drive.mount('/content/drive', timeout_ms=DRIVE_MOUNT_TIMEOUT_MS)
    else:
        print('Colab Drive mount skipped. Runtime-local paths under /content will be used.')
        print('Set MOUNT_DRIVE=True only when the Colab Drive auth prompt is available.')
elif RUN_MODE == "local":
    print('Local mode: Google Drive mount skipped.')



Local mode: Google Drive mount skipped.


## 2. Resolve Repository, Data, Bundle, and Output Paths

This cell converts the runtime-mode settings into canonical path variables used by the rest of the notebook: `REPO_ROOT`, `DATA_DIR`, `BUNDLE_PATH`, `OUTPUT_DIR`, `EMBEDDINGS_DIR`, `RING_EXE_PATH`, `RING_EDGE_OUTPUT_DIR`, and `PRECOMPUTED_RING_EDGES_DIR`.

`RUN_MODE="auto"` resolves to Colab when the active kernel is a Colab runtime, including when launched through PyCharm's Google Colab integration. In local mode, the notebook assumes the repository already exists. In Colab mode, the dependency cell can clone the repository into `/content` when needed. If Drive mounting is skipped, Colab data defaults to runtime-local `/content/deepmzyme_data` instead of `/content/drive`.


In [85]:
#@title Resolve paths and run environment checks
from pathlib import Path
import os
import sys


def _path(value, *, resolve=False):
    path = Path(str(value)).expanduser()
    return path.resolve() if resolve else path

if RUN_MODE == "local":
    REPO_ROOT = _path(LOCAL_REPO_ROOT, resolve=True)
    DATA_DIR = _path(LOCAL_DATA_DIR, resolve=True)
    OUTPUT_DIR = _path(LOCAL_OUTPUT_DIR, resolve=True)
    UNPACK_DIR = _path(LOCAL_UNPACK_DIR, resolve=True)
    BUNDLE_PATH = _path(LOCAL_BUNDLE_PATH, resolve=True) if str(LOCAL_BUNDLE_PATH).strip() else DATA_DIR / 'DeepMzyme_Colab_Bundles' / BUNDLE_FILENAME
    EMBEDDINGS_DIR = _path(LOCAL_EMBEDDINGS_DIR, resolve=True)
    RING_EXE_PATH = _path(LOCAL_RING_EXE_PATH, resolve=True) if str(LOCAL_RING_EXE_PATH).strip() else Path('')
    RING_EDGE_OUTPUT_DIR = _path(LOCAL_RING_EDGE_OUTPUT_DIR, resolve=True)
    PRECOMPUTED_RING_EDGES_DIR = _path(LOCAL_PRECOMPUTED_RING_EDGES_DIR, resolve=True)
    DRIVE_OUTPUT_DIR = OUTPUT_DIR
elif RUN_MODE == "colab":
    data_source = globals().get('COLAB_DATA_SOURCE', 'huggingface_link')
    drive_requested = data_source == 'drive' and bool(globals().get('MOUNT_DRIVE', False))
    drive_available = Path('/content/drive/MyDrive').exists()
    use_drive_paths = drive_requested and drive_available
    if data_source == 'drive' and not use_drive_paths:
        print('Warning: COLAB_DATA_SOURCE="drive", but /content/drive/MyDrive is not available. Mount Drive or choose upload_file/huggingface_link.')
    REPO_ROOT = _path(COLAB_REPO_ROOT, resolve=True)
    DATA_DIR = _path(COLAB_DRIVE_DATA_DIR if use_drive_paths else COLAB_LOCAL_DATA_DIR)
    OUTPUT_DIR = _path(COLAB_OUTPUT_DIR, resolve=True)
    UNPACK_DIR = _path(COLAB_UNPACK_DIR, resolve=True)
    if str(COLAB_BUNDLE_PATH).strip():
        BUNDLE_PATH = _path(COLAB_BUNDLE_PATH)
    elif data_source == 'drive':
        BUNDLE_PATH = DATA_DIR / 'DeepMzyme_Colab_Bundles' / BUNDLE_FILENAME
    else:
        BUNDLE_PATH = Path('/content') / BUNDLE_FILENAME
    EMBEDDINGS_DIR = _path(COLAB_EMBEDDINGS_DIR if use_drive_paths else COLAB_LOCAL_EMBEDDINGS_DIR)
    RING_EXE_PATH = _path(COLAB_RING_EXE_PATH)
    RING_EDGE_OUTPUT_DIR = _path(COLAB_RING_EDGE_OUTPUT_DIR if use_drive_paths else COLAB_LOCAL_RING_EDGE_OUTPUT_DIR)
    PRECOMPUTED_RING_EDGES_DIR = _path(COLAB_PRECOMPUTED_RING_EDGES_DIR if use_drive_paths else COLAB_LOCAL_PRECOMPUTED_RING_EDGES_DIR)
    DRIVE_OUTPUT_DIR = DATA_DIR / DRIVE_OUTPUT_SUBDIR if use_drive_paths else OUTPUT_DIR
else:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')

# Backward-compatible aliases used by later notebook helpers.
repo_dir = REPO_ROOT
repo_root = REPO_ROOT
REPO_DIR = str(REPO_ROOT)
drive_data_dir = DATA_DIR
bundle_path = BUNDLE_PATH
unpack_dir = UNPACK_DIR
local_runs_dir = OUTPUT_DIR / 'runs'
drive_output_dir = DRIVE_OUTPUT_DIR

DEFAULT_ESM_EMBEDDINGS_DIR = str(EMBEDDINGS_DIR)
DEFAULT_RING_EXE_PATH = str(RING_EXE_PATH)
DEFAULT_RING_EDGE_OUTPUT_DIR = str(RING_EDGE_OUTPUT_DIR)
DEFAULT_PRECOMPUTED_RING_EDGES_DIR = str(PRECOMPUTED_RING_EDGES_DIR)

required_files = [
    REPO_ROOT / 'AGENTS.md',
    REPO_ROOT / 'Plan.md',
    REPO_ROOT / 'README.md',
    REPO_ROOT / 'src' / 'train.py',
    REPO_ROOT / 'src' / 'report_runs.py',
]

print('Current working directory:', Path.cwd())
print('Python executable:', sys.executable)
print('RUN_MODE:', RUN_MODE)
print('REPO_ROOT:', REPO_ROOT)
print('REPO_ROOT exists:', REPO_ROOT.exists())
if RUN_MODE == 'colab':
    print('COLAB_DATA_SOURCE:', data_source)
    print('Using Google Drive paths:', use_drive_paths)
print('DATA_DIR:', DATA_DIR)
print('DATA_DIR exists:', DATA_DIR.exists())
print('BUNDLE_PATH:', BUNDLE_PATH)
print('BUNDLE_PATH exists:', BUNDLE_PATH.exists())
print('UNPACK_DIR:', UNPACK_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('EMBEDDINGS_DIR:', EMBEDDINGS_DIR)
print('EMBEDDINGS_DIR exists:', EMBEDDINGS_DIR.exists())
print('RING_EXE_PATH:', RING_EXE_PATH if str(RING_EXE_PATH) else '(not configured)')
if str(RING_EXE_PATH):
    print('RING_EXE_PATH exists:', RING_EXE_PATH.exists())
    print('RING_EXE_PATH executable:', os.access(RING_EXE_PATH, os.X_OK) if RING_EXE_PATH.exists() else False)
print('RING_EDGE_OUTPUT_DIR:', RING_EDGE_OUTPUT_DIR)
print('PRECOMPUTED_RING_EDGES_DIR:', PRECOMPUTED_RING_EDGES_DIR)
print('Required files:')
for path in required_files:
    print(' ', path, 'exists:', path.exists())



Current working directory: /home/mechti/PycharmProjects/DeepMzyme/notebooks
Python executable: /home/mechti/miniconda3/envs/DeepMzyme/bin/python
RUN_MODE: local
REPO_ROOT: /home/mechti/PycharmProjects/DeepMzyme
REPO_ROOT exists: True
DATA_DIR: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data
DATA_DIR exists: True
BUNDLE_PATH: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/DeepMzyme_Colab_Bundles/DeepMzyme_Data_runtime_local_2026-05-03.tar.zst
BUNDLE_PATH exists: False
UNPACK_DIR: /home/mechti/PycharmProjects/DeepMzyme/.notebook_bundle
OUTPUT_DIR: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/notebook_outputs
EMBEDDINGS_DIR: /media/Data/DeepMzyme_Data/esm_embeddings
EMBEDDINGS_DIR exists: True
RING_EXE_PATH: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/ring-4.0/out/bin/ring
RING_EXE_PATH exists: True
RING_EXE_PATH executable: True
RING_EDGE_OUTPUT_DIR: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/RING_features
PRECOMPUTED_RING_EDGES_DIR: /home/mecht

## 3. Repository and Dependency Checks

Colab mode can install dependencies in a fresh runtime. Local mode does not pip-install automatically; use the active Jupyter/PyCharm kernel from the DeepMzyme conda environment, or explicitly set `INSTALL_REQUIREMENTS=True` if you want this notebook to install into that kernel.

ESM embedding generation needs the optional ESMC package. Leave `INSTALL_ESM_DEPENDENCIES=False` when using Only-GVP or already-precomputed embeddings; enable it before setting `PREPARE_MISSING_ESM_EMBEDDINGS=True`.


In [86]:
#@title Repository and dependency checks
import os
import subprocess
import sys
from pathlib import Path

INSTALL_REQUIREMENTS = RUN_MODE == "colab"  #@param {type:"boolean"}
INSTALL_ESM_DEPENDENCIES = False  #@param {type:"boolean"}
CHECK_IMPORTS = True  #@param {type:"boolean"}

if RUN_MODE == "colab":
    if not REPO_GIT_URL or '<' in REPO_GIT_URL:
        raise ValueError('Set REPO_GIT_URL to your DeepMzyme repository URL before cloning.')
    if repo_dir.exists():
        print(f'Repository already exists at {repo_dir}; skipping clone.')
        print('Warning: an existing checkout may be stale or may contain local work. This notebook will not overwrite it automatically.')
        for git_cmd in (['git', 'branch', '--show-current'], ['git', 'rev-parse', 'HEAD'], ['git', 'status', '--short']):
            label = ' '.join(git_cmd)
            result = subprocess.run(git_cmd, cwd=repo_dir, check=False, capture_output=True, text=True)
            output = result.stdout.strip() or result.stderr.strip() or '(no output)'
            print(f'$ {label}\\n{output}')
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_GIT_BRANCH, REPO_GIT_URL, str(repo_dir)], check=True)
elif RUN_MODE == "local":
    if not repo_dir.exists():
        hint = ''
        if _running_in_colab():
            hint = ' The active kernel appears to be Colab, so your PC path is not visible there. Set RUN_MODE to auto or colab, then rerun from the first cell so the notebook clones into COLAB_REPO_ROOT.'
        raise FileNotFoundError(f'Local REPO_ROOT does not exist: {repo_dir}.{hint}')
    print(f'Local repository: {repo_dir}')
    for git_cmd in (['git', 'branch', '--show-current'], ['git', 'rev-parse', 'HEAD'], ['git', 'status', '--short']):
        label = ' '.join(git_cmd)
        result = subprocess.run(git_cmd, cwd=repo_dir, check=False, capture_output=True, text=True)
        output = result.stdout.strip() or result.stderr.strip() or '(no output)'
        print(f'$ {label}\\n{output}')

requirements_path = repo_dir / 'src' / 'requirements.txt'
if INSTALL_REQUIREMENTS:
    if requirements_path.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    else:
        if RUN_MODE != "colab":
            raise FileNotFoundError(f'Requirements file not found: {requirements_path}')
        subprocess.run([
            sys.executable, '-m', 'pip', 'install',
            'torch-geometric==2.7.0', 'biopython==1.87', 'biotite>=1.6', 'propka>=3.5', 'gemmi>=0.7', 'pandas', 'matplotlib'
        ], check=True)
else:
    print('Dependency installation skipped. Set INSTALL_REQUIREMENTS=True to install into the active notebook kernel.')

if INSTALL_ESM_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'esm'], check=True)
else:
    print('Optional ESMC dependency installation skipped. Enable INSTALL_ESM_DEPENDENCIES before generating missing ESM embeddings in Colab.')

if CHECK_IMPORTS:
    src_path = str(repo_dir / 'src')
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    import torch
    import pandas as pd
    import torch_geometric
    print('Python:', sys.executable)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('PyTorch Geometric:', torch_geometric.__version__)
    try:
        from esm.models.esmc import ESMC  # noqa: F401
    except Exception as exc:
        print('ESMC import: unavailable; precomputed ESM embeddings can still be used, but generation requires INSTALL_ESM_DEPENDENCIES=True. Detail:', exc)
    else:
        print('ESMC import: available')



Local repository: /home/mechti/PycharmProjects/DeepMzyme
$ git branch --show-current\nmain
$ git rev-parse HEAD\n2afdadf252ad4dc07194131fd7e9d003e5d7047e
$ git status --short\nM Plan.md
 M README.md
A  deepmzyme_notebook_fix_prompt.md
 M notebooks/DeepMzyme_training_colab.ipynb
 M src/data_structures.py
 M src/featurization.py
 M src/graph/construction.py
 M src/training/config.py
 M src/training/graph_dataset.py
 M src/training/preflight.py
 M src/training/run.py
 M src/training/splits.py
 M tests/smoke_checks.py
Dependency installation skipped. Set INSTALL_REQUIREMENTS=True to install into the active notebook kernel.
Optional ESMC dependency installation skipped. Enable INSTALL_ESM_DEPENDENCIES before generating missing ESM embeddings in Colab.
Python: /home/mechti/miniconda3/envs/DeepMzyme/bin/python
Torch: 2.11.0+cu130
CUDA available: False
PyTorch Geometric: 2.7.0
ESMC import: available


## 4. Optional Bundle Unpack

Colab mode unpacks the compressed bundle by default. Local mode skips unpacking by default and searches `DATA_DIR`, `UNPACK_DIR`, and `REPO_ROOT` for the dataset; enable `UNPACK_BUNDLE` if you want to extract a local bundle.


In [87]:
#@title Optional bundle unpack
import hashlib
import shutil
import subprocess
import urllib.request
from pathlib import Path

UNPACK_BUNDLE = RUN_MODE == "colab"  #@param {type:"boolean"}
UPLOAD_LOCAL_BUNDLE_IN_COLAB = RUN_MODE == "colab" and COLAB_DATA_SOURCE == "upload_file"  #@param {type:"boolean"}
DOWNLOAD_HUGGINGFACE_BUNDLE_IN_COLAB = RUN_MODE == "colab" and COLAB_DATA_SOURCE == "huggingface_link"  #@param {type:"boolean"}
FORCE_REUNPACK = False  #@param {type:"boolean"}
INSTALL_ZSTD_IN_COLAB = True  #@param {type:"boolean"}


def run_checked(cmd):
    print('+', ' '.join(str(part) for part in cmd))
    subprocess.run([str(part) for part in cmd], check=True)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def verify_sha256(path: Path, expected_sha256: str) -> None:
    expected = str(expected_sha256).strip().lower()
    if not expected:
        return
    observed = sha256_file(path)
    if observed.lower() != expected:
        raise ValueError(f'SHA256 mismatch for {path}: expected {expected}, observed {observed}')
    print('Verified SHA256:', observed)


def download_file(url: str, destination: Path, expected_sha256: str = '') -> Path:
    if not str(url).strip():
        raise ValueError('Hugging Face bundle URL is empty.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not FORCE_REUNPACK:
        print('Using existing downloaded bundle:', destination)
        verify_sha256(destination, expected_sha256)
        return destination
    if destination.exists():
        destination.unlink()
    print('Downloading bundle:', url)
    print('Destination:', destination)
    urllib.request.urlretrieve(url, destination)
    verify_sha256(destination, expected_sha256)
    return destination


def unpack_archive(source: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(source.suffixes).lower()
    if suffixes.endswith('.tar.zst'):
        if shutil.which('zstd') is None:
            if RUN_MODE == "colab" and INSTALL_ZSTD_IN_COLAB:
                run_checked(['apt-get', 'update'])
                run_checked(['apt-get', 'install', '-y', 'zstd'])
            else:
                raise RuntimeError('zstd is required to unpack .tar.zst bundles. Install zstd or set UNPACK_BUNDLE=False.')
        run_checked(['tar', '--use-compress-program=zstd', '-xf', source, '-C', destination])
    elif suffixes.endswith(('.tar.gz', '.tgz', '.tar')):
        run_checked(['tar', '-xf', source, '-C', destination])
    else:
        raise ValueError(f'Unsupported bundle archive type: {source}')


def adopt_unpacked_runtime_data_dir() -> None:
    global DATA_DIR, drive_data_dir, EMBEDDINGS_DIR, RING_EDGE_OUTPUT_DIR, PRECOMPUTED_RING_EDGES_DIR
    global DEFAULT_ESM_EMBEDDINGS_DIR, DEFAULT_RING_EDGE_OUTPUT_DIR, DEFAULT_PRECOMPUTED_RING_EDGES_DIR
    if RUN_MODE != "colab" or globals().get('use_drive_paths', False):
        return
    candidates = [unpack_dir / 'DeepMzyme_Data', unpack_dir / 'deepmzyme_data']
    runtime_data_dir = next((candidate for candidate in candidates if candidate.is_dir()), None)
    if runtime_data_dir is None:
        return
    DATA_DIR = runtime_data_dir.resolve()
    drive_data_dir = DATA_DIR
    if (DATA_DIR / 'esm_embeddings').is_dir():
        EMBEDDINGS_DIR = DATA_DIR / 'esm_embeddings'
        DEFAULT_ESM_EMBEDDINGS_DIR = str(EMBEDDINGS_DIR)
    if (DATA_DIR / 'RING_features').is_dir():
        RING_EDGE_OUTPUT_DIR = DATA_DIR / 'RING_features'
        PRECOMPUTED_RING_EDGES_DIR = DATA_DIR / 'RING_features'
        DEFAULT_RING_EDGE_OUTPUT_DIR = str(RING_EDGE_OUTPUT_DIR)
        DEFAULT_PRECOMPUTED_RING_EDGES_DIR = str(PRECOMPUTED_RING_EDGES_DIR)
    print('Runtime-local DATA_DIR adopted from unpacked archive:', DATA_DIR)
    print('Runtime-local EMBEDDINGS_DIR:', EMBEDDINGS_DIR)
    print('Runtime-local RING features dir:', RING_EDGE_OUTPUT_DIR)


if RUN_MODE == "colab" and DOWNLOAD_HUGGINGFACE_BUNDLE_IN_COLAB:
    bundle_path = download_file(COLAB_HUGGINGFACE_BUNDLE_URL, Path(BUNDLE_PATH), COLAB_HUGGINGFACE_BUNDLE_SHA256)
    BUNDLE_PATH = bundle_path
    UNPACK_BUNDLE = True

if RUN_MODE == "colab" and UPLOAD_LOCAL_BUNDLE_IN_COLAB and not bundle_path.exists():
    from google.colab import files
    print('Upload the local .tar.zst bundle now. BUNDLE_PATH will be updated to the uploaded file.')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No bundle was uploaded.')
    uploaded_name = next(iter(uploaded))
    uploaded_path = Path('/content') / uploaded_name
    bundle_path = uploaded_path
    BUNDLE_PATH = uploaded_path
    verify_sha256(uploaded_path, globals().get('COLAB_HUGGINGFACE_BUNDLE_SHA256', ''))
    UNPACK_BUNDLE = True

if UNPACK_BUNDLE:
    if FORCE_REUNPACK and unpack_dir.exists():
        shutil.rmtree(unpack_dir)

    if bundle_path.is_dir():
        unpack_dir = bundle_path.resolve()
        UNPACK_DIR = unpack_dir
        print(f'Using already-unpacked bundle directory: {unpack_dir}')
        adopt_unpacked_runtime_data_dir()
    else:
        if not bundle_path.exists():
            raise FileNotFoundError(f'Bundle not found: {bundle_path}. If Drive mount is disabled, upload or copy the bundle under DATA_DIR / DeepMzyme_Colab_Bundles, or set BUNDLE_PATH to an accessible location and rerun this cell.')
        marker = unpack_dir / '.deepmzyme_bundle_unpacked'
        if marker.exists() and not FORCE_REUNPACK:
            print(f'Bundle already unpacked at {unpack_dir}. Set FORCE_REUNPACK=True to extract again.')
        else:
            unpack_archive(bundle_path, unpack_dir)
            marker.write_text('ok\\n', encoding='utf-8')
            print(f'Unpacked bundle into {unpack_dir}')
        adopt_unpacked_runtime_data_dir()
else:
    print('Bundle unpack skipped. Dataset discovery will search UNPACK_DIR, DATA_DIR, and REPO_ROOT.')



Bundle unpack skipped. Dataset discovery will search UNPACK_DIR, DATA_DIR, and REPO_ROOT.


## 5. Verify Train/Test Directories and Summary CSVs

DeepMzyme training needs a train structure directory, a test structure directory, and MAHOMES-style summary CSVs with site-level columns such as PDB ID, EC number, metal residue number, and metal residue type. The bundle may also contain structure-level CSV artifacts; those are useful metadata, but they are not used here unless they match the training loader requirements.

In [88]:
#@title Detect and verify dataset paths
import csv
from pathlib import Path

TRAIN_DIR_OVERRIDE = ''  #@param {type:"string"}
TEST_DIR_OVERRIDE = ''  #@param {type:"string"}
TRAIN_CSV_OVERRIDE = ''  #@param {type:"string"}
TEST_CSV_OVERRIDE = ''  #@param {type:"string"}

REQUIRED_SUMMARY_ALIASES = {
    'pdbid': {'pdbid', 'structure'},
    'metal residue number': {'metal residue number', 'chain_resi'},
    'EC number': {'ec number', 'ecnumber'},
    'metal residue type': {'metal residue type', 'metaltype'},
}


def has_required_summary_columns(path: Path) -> bool:
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = {field.strip().lower() for field in (reader.fieldnames or []) if field}
    except Exception:
        return False
    return all(fields.intersection(aliases) for aliases in REQUIRED_SUMMARY_ALIASES.values())


def validate_site_level_summary_csv(path: Path, split_name: str) -> None:
    if has_required_summary_columns(path):
        return
    try:
        with path.open('r', encoding='utf-8', newline='') as handle:
            reader = csv.DictReader(handle)
            fields = [field.strip() for field in (reader.fieldnames or []) if field]
    except Exception as exc:
        raise ValueError(f'Could not read {split_name} summary CSV {path}: {exc}') from exc
    raise ValueError(
        f'{split_name} summary CSV is not a site-level MAHOMES training summary: {path}. '
        f'Observed columns: {fields}. Structure-level metadata such as '
        'structure_name/ec_numbers/metal_type is not sufficient for training; rebuild the Colab bundle with the site-level summary CSVs.'
    )


def find_dataset_root(root: Path, dataset_name: str) -> Path:
    root = Path(root).expanduser()
    candidates = [
        root / 'DeepMzyme_Data' / dataset_name,
        root / dataset_name,
        root,
    ]
    if root.exists():
        candidates.extend(path for path in root.rglob(dataset_name) if path.is_dir())
    for candidate in candidates:
        if (candidate / 'train').is_dir() and (candidate / 'test').is_dir():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find train/test directories for dataset {dataset_name!r} under {root}')


def find_dataset_root_in_roots(roots, dataset_name: str) -> Path:
    errors = []
    for root in roots:
        try:
            return find_dataset_root(Path(root), dataset_name)
        except FileNotFoundError as exc:
            errors.append(str(exc))
    raise FileNotFoundError('Could not find dataset root. Checked:\\n- ' + '\\n- '.join(errors))


def find_summary_csv(split_dir: Path, split_name: str) -> Path:
    preferred_names = [
        'catalytic_only_summary.csv',
        'final_data_summarazing_table_transition_metals_only_catalytic.csv',
        f'{DATASET_NAME}_{split_name}.csv',
    ]
    candidates = [split_dir / name for name in preferred_names]
    candidates.extend(sorted(split_dir.glob('*.csv')))
    for root in [unpack_dir, DATA_DIR, REPO_ROOT]:
        root = Path(root)
        if root.exists():
            candidates.extend(sorted(root.glob(f'**/*{split_name}*.csv')))
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)
    for candidate in unique_candidates:
        if candidate.exists() and has_required_summary_columns(candidate):
            return candidate.resolve()
    found = [str(path) for path in unique_candidates if path.exists()]
    raise FileNotFoundError(
        f'No MAHOMES-style {split_name} summary CSV found. Checked: {found}. '
        'If your bundle only contains structure_name/ec_numbers/metal_type CSVs, rebuild it with the site-level summary CSV included.'
    )

search_roots = [unpack_dir, DATA_DIR, REPO_ROOT]
dataset_root = find_dataset_root_in_roots(search_roots, DATASET_NAME)
train_dir = Path(TRAIN_DIR_OVERRIDE).expanduser().resolve() if TRAIN_DIR_OVERRIDE else dataset_root / 'train'
test_dir = Path(TEST_DIR_OVERRIDE).expanduser().resolve() if TEST_DIR_OVERRIDE else dataset_root / 'test'
train_csv = Path(TRAIN_CSV_OVERRIDE).expanduser().resolve() if TRAIN_CSV_OVERRIDE else find_summary_csv(train_dir, 'train')
test_csv = Path(TEST_CSV_OVERRIDE).expanduser().resolve() if TEST_CSV_OVERRIDE else find_summary_csv(test_dir, 'test')

for label, path in [('train_dir', train_dir), ('test_dir', test_dir), ('train_csv', train_csv), ('test_csv', test_csv)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} does not exist: {path}')
validate_site_level_summary_csv(train_csv, 'train')
validate_site_level_summary_csv(test_csv, 'test')

train_structures = sorted(list(train_dir.glob('*.pdb')) + list(train_dir.glob('*.cif')) + list(train_dir.glob('*.mmcif')))
test_structures = sorted(list(test_dir.glob('*.pdb')) + list(test_dir.glob('*.cif')) + list(test_dir.glob('*.mmcif')))
if not train_structures or not test_structures:
    raise ValueError(f'Expected structure files in both train and test directories: {train_dir}, {test_dir}')

print('Dataset search roots:', [str(root) for root in search_roots])
print('Dataset root:', dataset_root)
print('Train structures:', len(train_structures), train_dir)
print('Test structures:', len(test_structures), test_dir)
print('Train summary CSV:', train_csv)
print('Test summary CSV:', test_csv)



Dataset search roots: ['/home/mechti/PycharmProjects/DeepMzyme/.notebook_bundle', '/home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data', '/home/mechti/PycharmProjects/DeepMzyme']
Dataset root: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/train_and_test_sets_structures_non_overlapped_pinmymetal
Train structures: 1304 /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/train_and_test_sets_structures_non_overlapped_pinmymetal/train
Test structures: 316 /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/train_and_test_sets_structures_non_overlapped_pinmymetal/test
Train summary CSV: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/train_and_test_sets_structures_non_overlapped_pinmymetal/train/final_data_summarazing_table_transition_metals_only_catalytic.csv
Test summary CSV: /home/mechti/PycharmProjects/DeepMzyme/DeepMzyme_Data/train_and_test_sets_structures_non_overlapped_pinmymetal/test/final_data_summarazing_table_transition_metals_only_catalytic.csv


## 6. Set Default Run Values

These values provide defaults for the experiment grid and keep the notebook runnable without the optional widget panel. The widget panel below is the preferred interface: one selected value per grid field gives one run; multiple selected values or CSV entries create a sweep. `MAX_SWEEP_RUNS` protects against accidental huge Cartesian products.

Use `RUN_PROFILE='quick_smoke'` for a one-epoch CPU path check before a real run. Use `RUN_PROFILE='normal_single_run'` for the default baseline settings, or `custom` when you want the explicit fields below to be used exactly as written.

Model preset meanings:

- `Only-GVP` / `only_gvp`: GVP graph-only baseline, no ESM branch and no ESM embeddings required.
- `Only-ESM` / `only_esm`: ESM embedding baseline without structural graph message passing; needs precomputed ESM embeddings unless missing-embedding preparation is intentionally enabled.
- `GVP + late fusion` / `gvp + late_fusion`: first simple combined graph + ESM baseline.
- `GVP + early fusion` / `gvp + early_fusion`: residue/node-level ESM input to the GVP stack.
- `GVP + node-level late fusion` / `gvp + node_level_late_fusion`: node-level ESM/graph fusion before pooling.
- `GVP + hybrid fusion` / `gvp + hybrid`: combined early and late ESM usage.
- `GVP + cross-modal attention` / `gvp + cross_modal_attention`: advanced fusion; do not include in the default first-stage run.
- `SimpleGNN + ESM` / `simple_gnn_esm`: simpler non-GVP graph + ESM comparison.

Task meanings:

- `metal_6_class`: metal task with `--task metal`, selected by validation metal balanced accuracy.
- `ec_prediction`: EC task with `--task ec`, selected by validation EC group balanced accuracy.
- `joint_metal_ec`: joint metal + EC task with `--task joint`, selected by validation joint balanced accuracy.
- `metal_collapsed4_metric`: metal task selected with the collapsed-4 validation metric for that specific comparison.

`fusion_mode` matters only when graph/GVP and ESM are combined. `Only-GVP` should not require ESM embeddings. `Only-ESM`, `SimpleGNN + ESM`, and GVP+ESM runs need precomputed ESM embeddings unless `PREPARE_MISSING_ESM_EMBEDDINGS` is deliberately enabled. In local mode, `LOCAL_EMBEDDINGS_DIR` defaults to `DEEPMZYME_ESM_EMBEDDINGS_DIR` when that environment variable is set, otherwise `/media/Data/DeepMzyme_Data/esm_embeddings`; edit the field for other systems.

Recommended staged search:

0. Stage 0 readiness/debug: 1 epoch, small/simple model, checks paths/data/embeddings/output only.
1. Stage 1 baselines: `only_gvp`, `only_esm`, `gvp + late_fusion`.
2. Stage 2 GVP capacity check: vary `gvp_layers`, `hidden_s`, `hidden_v`, and `edge_hidden`; keep LR fixed first.
3. Stage 3 edge-radius check: vary `edge_radius` after choosing a reasonable capacity.
4. Stage 4 ESM fusion comparison: `late_fusion`, `early_fusion`, `node_level_late_fusion`, `hybrid`; add `cross_modal_attention` later only if simpler fusion helps.
5. Stage 5 RING: try RING only after the radius-only baseline is stable.

Learning-rate guidance: readiness/debug runs can use `3e-5`, `1e-4`, or `3e-4` because quality is not being judged. For the serious first baseline comparison, prefer `3e-5` with `weight_decay=1e-4`, based on previous DeepMzyme results among tested values. Treat `1e-4` as the main follow-up confirmation value; if time is limited, compare only `3e-5` and `1e-4`. Include `3e-4` only as an optional debug/wider comparison, not as the main serious baseline recommendation. Use validation metrics for LR/model choice, not repeated held-out test checks.

GPU memory guidance (Colab T4, about 15 GB):

- Safe baseline: `gvp_layers=4`, `hidden_s=128`, `hidden_v=16`, `edge_hidden=64`, `batch_size=8`.
- Approaching limit: `hidden_s=192` or `hidden_v=32`; reduce `batch_size` to 4 if OOM.
- `gvp_layers >= 6` with `hidden_s=192` is likely OOM on T4 at `batch_size=8`; use `batch_size=4` or smaller.
- `Only-ESM` uses much less VRAM than GVP variants.

Do not combine all GVP sizes, edge radii, fusion modes, learning rates, seeds, EC depths, and RING modes in one huge sweep. Build staged grids with one or two varying dimensions at a time.

Use validation metrics for model, checkpoint, and hyperparameter choice. Use held-out test metrics only for final reporting of validation-selected runs.

`RING edges mode` controls extra RING interaction edges on top of the normal radius graph:

- `Radius only (no RING interaction edges)`: do not pass `--use-ring-edges`; this is the safest baseline.
- `Radius + existing RING edge files`: pass precomputed RING edge files from `PRECOMPUTED_RING_EDGES_DIR`.
- `Radius + generate missing RING edge files`: ask training/preflight logic to generate missing RING edge files into `RING_EDGE_OUTPUT_DIR`.


In [89]:
#@title Default run values
from datetime import datetime
from pathlib import Path

RING_EDGE_MODE_LABEL_TO_VALUE = {
    'Radius only (no RING interaction edges)': 'radius_only',
    'Radius + existing RING edge files': 'radius_plus_precomputed_ring',
    'Radius + generate missing RING edge files': 'generate_missing_ring',
}
RING_EDGE_MODE_VALUE_TO_LABEL = {value: label for label, value in RING_EDGE_MODE_LABEL_TO_VALUE.items()}

CONSERVATIVE_NODE_FEATURES = [
    'aa_one_hot',
    'hydrophobicity_kd',
    'donor_flag',
    'acceptor_flag',
    'aromatic_flag',
    'acidic_flag',
    'basic_flag',
    'is_first_shell',
    'is_second_shell',
    'ca_to_metal',
    'fg_to_metal',
    'min_donor_to_metal',
    'biotite_residue_sasa',
    'custom_charge_distance_proxy',
    'dpka_titr',
    'v_cb_to_fg',
    'v_res_to_metal',
    'cos_theta_between_vnetligand_to_vrestometal',
]


def normalize_ring_edges_mode(value):
    text = str(value).strip()
    if text in RING_EDGE_MODE_LABEL_TO_VALUE:
        return RING_EDGE_MODE_LABEL_TO_VALUE[text]
    if text in RING_EDGE_MODE_VALUE_TO_LABEL:
        return text
    valid = ', '.join(RING_EDGE_MODE_LABEL_TO_VALUE)
    raise ValueError(f'Unsupported RING edge mode {value!r}. Choose one of: {valid}')


def ring_edges_mode_label(value):
    mode = normalize_ring_edges_mode(value)
    return RING_EDGE_MODE_VALUE_TO_LABEL[mode]

TASK_MODE = 'metal_6_class'  #@param ['metal_6_class', 'metal_collapsed4_metric', 'ec_prediction', 'joint_metal_ec']
MODEL_PRESET = 'Only-GVP'  #@param ['Only-GVP', 'Only-ESM', 'GVP + late fusion', 'GVP + early fusion', 'GVP + node-level late fusion', 'GVP + hybrid fusion', 'GVP + cross-modal attention', 'SimpleGNN + ESM']
RUN_PROFILE = 'normal_single_run'  #@param ['quick_smoke', 'normal_single_run', 'custom']

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 8  #@param {type:"integer"}
LEARNING_RATE = 3e-5  #@param {type:"number"}
WEIGHT_DECAY = 1e-4  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}
VAL_FRACTION = 0.15  #@param {type:"number"}
DEVICE = 'cuda'  #@param ['cuda', 'cpu']
RUN_NAME = ''  #@param {type:"string"}

GVP_LAYERS = 4  #@param {type:"integer"}
HIDDEN_S = 128  #@param {type:"integer"}
HIDDEN_V = 16  #@param {type:"integer"}
EDGE_HIDDEN = 64  #@param {type:"integer"}
EDGE_RADIUS = 8.0  #@param {type:"number"}
ESM_FUSION_DIM = 128  #@param {type:"integer"}
HEAD_MLP_LAYERS = 2  #@param {type:"integer"}

EC_LABEL_DEPTH = 1  #@param {type:"integer"}
EC_CONTRASTIVE_WEIGHT = 0.0  #@param {type:"number"}
# Conservative is the only named node feature set; omit individual sub-features below if needed.
NODE_FEATURE_SET = 'conservative'
NODE_FEATURES_OMIT = []
SPLIT_BY = 'pdbid'  #@param ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']
LR_SCHEDULE = 'fixed'  #@param ['fixed', 'cosine', 'step']
LR_STEP_SIZE = 10  #@param {type:"integer"}
LR_DECAY_GAMMA = 0.5  #@param {type:"number"}

CROSS_ATTENTION_LAYERS = 1  #@param {type:"integer"}
CROSS_ATTENTION_HEADS = 4  #@param {type:"integer"}
CROSS_ATTENTION_LAYERS = [CROSS_ATTENTION_LAYERS]
CROSS_ATTENTION_HEADS = [CROSS_ATTENTION_HEADS]
CROSS_ATTENTION_DROPOUT = 0.1  #@param {type:"number"}
CROSS_ATTENTION_NEIGHBORHOOD = 'all'  #@param ['all', 'first_shell', 'first_second_shell']
CROSS_ATTENTION_BIDIRECTIONAL = False  #@param {type:"boolean"}
# Reference Colab/default literals used by smoke checks and documentation.
# ESM_EMBEDDINGS_DIR = '/content/drive/MyDrive/DeepMzyme/DeepMzyme_Data/esm_embeddings'
# RING_EDGES_MODE = 'radius_only'
# RING_EXE_PATH = 'DeepMzyme_Data/ring-4.0/out/bin/ring'

USE_ESM_EMBEDDINGS = True  #@param {type:"boolean"}
ESM_EMBEDDINGS_DIR = DEFAULT_ESM_EMBEDDINGS_DIR  #@param {type:"string"}
PREPARE_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_ESM_EMBEDDINGS = False  #@param {type:"boolean"}
ALLOW_MISSING_EXTERNAL_FEATURES = True  #@param {type:"boolean"}

RING_EDGES_MODE = 'Radius only (no RING interaction edges)'  #@param ['Radius only (no RING interaction edges)', 'Radius + existing RING edge files', 'Radius + generate missing RING edge files']
RING_EXE_PATH = DEFAULT_RING_EXE_PATH  #@param {type:"string"}
RING_EDGE_OUTPUT_DIR = DEFAULT_RING_EDGE_OUTPUT_DIR  #@param {type:"string"}
PRECOMPUTED_RING_EDGES_DIR = DEFAULT_PRECOMPUTED_RING_EDGES_DIR  #@param {type:"string"}
REQUIRE_RING_EDGES = False  #@param {type:"boolean"}
PREPARE_MISSING_RING_EDGES = False  #@param {type:"boolean"}
RING_EDGES_MODE = normalize_ring_edges_mode(RING_EDGES_MODE)

RUN_HELD_OUT_TEST_EVAL = True  #@param {type:"boolean"}

if RUN_PROFILE == 'quick_smoke':
    EPOCHS = 1
    BATCH_SIZE = min(int(BATCH_SIZE), 4)
    DEVICE = 'cpu'
    RUN_HELD_OUT_TEST_EVAL = False
    RUN_NAME = RUN_NAME or 'quick_smoke'
elif RUN_PROFILE not in {'normal_single_run', 'custom'}:
    raise ValueError(f'Unsupported RUN_PROFILE: {RUN_PROFILE}')

task_map = {
    'metal_6_class': ('metal', 'val_metal_balanced_acc'),
    'metal_collapsed4_metric': ('metal', 'val_metal_collapsed4_balanced_acc'),
    'ec_prediction': ('ec', 'val_ec_group_balanced_acc'),
    'joint_metal_ec': ('joint', 'val_joint_balanced_acc'),
}
preset_map = {
    'Only-GVP': {'model_architecture': 'only_gvp', 'uses_esm': False},
    'Only-ESM': {'model_architecture': 'only_esm', 'uses_esm': True},
    'GVP + late fusion': {'model_architecture': 'gvp', 'fusion_mode': 'late_fusion', 'uses_esm': True},
    'GVP + early fusion': {'model_architecture': 'gvp', 'fusion_mode': 'early_fusion', 'uses_esm': True},
    'GVP + node-level late fusion': {'model_architecture': 'gvp', 'fusion_mode': 'node_level_late_fusion', 'uses_esm': True},
    'GVP + hybrid fusion': {'model_architecture': 'gvp', 'fusion_mode': 'hybrid', 'uses_esm': True},
    'GVP + cross-modal attention': {'model_architecture': 'gvp', 'fusion_mode': 'cross_modal_attention', 'uses_esm': True},
    'SimpleGNN + ESM': {'model_architecture': 'simple_gnn_esm', 'fusion_mode': 'late_fusion', 'uses_esm': True},
}

task_name, selection_metric = task_map[TASK_MODE]
preset = preset_map[MODEL_PRESET]
print('Task:', task_name)
print('Selection metric:', selection_metric)
print('Model preset:', MODEL_PRESET)
print('Selected model uses ESM:', bool(preset.get('uses_esm')))
print('GVP/capacity controls:', {
    'gvp_layers': GVP_LAYERS,
    'hidden_s': HIDDEN_S,
    'hidden_v': HIDDEN_V,
    'edge_hidden': EDGE_HIDDEN,
    'edge_radius': EDGE_RADIUS,
    'esm_fusion_dim': ESM_FUSION_DIM,
    'head_mlp_layers': HEAD_MLP_LAYERS,
})
print('ESM embeddings enabled for ESM presets:', USE_ESM_EMBEDDINGS)
print('RING edge mode:', ring_edges_mode_label(RING_EDGES_MODE), f'[{RING_EDGES_MODE}]')
print('Run profile:', RUN_PROFILE)
print('Requested run name:', RUN_NAME or '(auto)')


Task: metal
Selection metric: val_metal_balanced_acc
Model preset: Only-GVP
Selected model uses ESM: False
GVP/capacity controls: {'gvp_layers': 4, 'hidden_s': 128, 'hidden_v': 16, 'edge_hidden': 64, 'edge_radius': 8.0, 'esm_fusion_dim': 128, 'head_mlp_layers': 2}
ESM embeddings enabled for ESM presets: True
RING edge mode: Radius only (no RING interaction edges) [radius_only]
Run profile: normal_single_run
Requested run name: (auto)


## 7. Configure Multi-Run Sweep Mode

Sweep mode expands the selected lists into planned experiments with `itertools.product`, then runs each plan through `src/train.py`. Use validation metrics first when comparing sweep results; held-out test metrics are final reporting for the validation-selected checkpoint only and should not be used to choose hyperparameters.

In [90]:
#@title Sweep configuration
# Keep these lists short until the full workflow is verified in Colab.
# One value in every list gives one run; multiple values create a Cartesian-product sweep.
# MAX_SWEEP_RUNS below prevents accidental huge sweeps.
TASK_MODES = ['metal_6_class']
MODEL_PRESETS = ['Only-GVP']
# Add more RING edge labels here only after the radius-only baseline is stable.
RING_EDGES_MODES = ['Radius only (no RING interaction edges)']
LEARNING_RATES = [LEARNING_RATE]
WEIGHT_DECAYS = [WEIGHT_DECAY]
BATCH_SIZES = [BATCH_SIZE]
SEEDS = [SEED]
NODE_FEATURE_SETS = [NODE_FEATURE_SET]
EC_LABEL_DEPTHS = [EC_LABEL_DEPTH]
EC_CONTRASTIVE_WEIGHTS = [EC_CONTRASTIVE_WEIGHT]
GVP_LAYER_VALUES = [GVP_LAYERS]
HIDDEN_S_VALUES = [HIDDEN_S]
HIDDEN_V_VALUES = [HIDDEN_V]
EDGE_HIDDEN_VALUES = [EDGE_HIDDEN]
EDGE_RADIUS_VALUES = [EDGE_RADIUS]
ESM_FUSION_DIM_VALUES = [ESM_FUSION_DIM]
HEAD_MLP_LAYER_VALUES = [HEAD_MLP_LAYERS]
CROSS_ATTENTION_LAYERS = list(CROSS_ATTENTION_LAYERS)
CROSS_ATTENTION_HEADS = list(CROSS_ATTENTION_HEADS)
LR_SCHEDULES = [LR_SCHEDULE]

MAX_SWEEP_RUNS = 24  #@param {type:"integer"}
STOP_ON_FIRST_FAILURE = True  #@param {type:"boolean"}
SKIP_EXISTING_RUNS = True  #@param {type:"boolean"}

print('Sweep task modes:', TASK_MODES)
print('Sweep model presets:', MODEL_PRESETS)
print('Available RING edge modes:', list(RING_EDGE_MODE_LABEL_TO_VALUE))
print('Sweep RING edge modes:', [ring_edges_mode_label(mode) for mode in RING_EDGES_MODES])
print('GVP/capacity sweep values:', {
    'gvp_layers': GVP_LAYER_VALUES,
    'hidden_s': HIDDEN_S_VALUES,
    'hidden_v': HIDDEN_V_VALUES,
    'edge_hidden': EDGE_HIDDEN_VALUES,
    'edge_radius': EDGE_RADIUS_VALUES,
    'esm_fusion_dim': ESM_FUSION_DIM_VALUES,
    'head_mlp_layers': HEAD_MLP_LAYER_VALUES,
})
print('Maximum planned runs:', MAX_SWEEP_RUNS)


Sweep task modes: ['metal_6_class']
Sweep model presets: ['Only-GVP']
Available RING edge modes: ['Radius only (no RING interaction edges)', 'Radius + existing RING edge files', 'Radius + generate missing RING edge files']
Sweep RING edge modes: ['Radius only (no RING interaction edges)']
GVP/capacity sweep values: {'gvp_layers': [4], 'hidden_s': [128], 'hidden_v': [16], 'edge_hidden': [64], 'edge_radius': [8.0], 'esm_fusion_dim': [128], 'head_mlp_layers': [2]}
Maximum planned runs: 24


## Optional Interactive Configuration Panel

The widget panel below is the preferred configuration interface. It keeps a safe default of preview-only execution and radius-only graph edges, while exposing advanced model, ESM, RING, loss, split, and reporting options.

One selected value in each sweep field creates a single planned run. Multiple selected values, or comma-separated values in the paired custom text boxes, create a guarded Cartesian-product sweep. Unsupported custom values are rejected before a command is built.


In [91]:
#@title Interactive configuration panel
from IPython.display import HTML, display

try:
    import ipywidgets as widgets
    _WIDGET_PANEL_AVAILABLE = True
except ModuleNotFoundError:
    _WIDGET_PANEL_AVAILABLE = False

    class _FallbackControl:
        def __init__(self, value=None, children=None, options=None):
            self.value = value
            self.children = children or []
            self.options = options or []
            self.layout = None

        def set_title(self, index, title):
            return None

        def add_class(self, class_name):
            return self

        def observe(self, handler, names=None):
            return None

    class _FallbackWidgets:
        def Layout(self, **kwargs):
            return None

        def HTML(self, value='', **kwargs):
            return _FallbackControl(value=value)

        def Dropdown(self, *, value='', options=(), **kwargs):
            return _FallbackControl(value=value, options=list(options))

        def Combobox(self, *, value='', options=(), **kwargs):
            return _FallbackControl(value=value, options=list(options))

        def IntText(self, *, value=0, **kwargs):
            return _FallbackControl(value=value)

        def FloatText(self, *, value=0.0, **kwargs):
            return _FallbackControl(value=value)

        def Text(self, *, value='', **kwargs):
            return _FallbackControl(value=value)

        def Checkbox(self, *, value=False, **kwargs):
            return _FallbackControl(value=value)

        def SelectMultiple(self, *, value=(), options=(), **kwargs):
            return _FallbackControl(value=tuple(value), options=list(options))

        def HBox(self, children, **kwargs):
            return _FallbackControl(children=list(children))

        def VBox(self, children, **kwargs):
            return _FallbackControl(children=list(children))

        def GridBox(self, children, **kwargs):
            return _FallbackControl(children=list(children))

        def Tab(self, children, **kwargs):
            return _FallbackControl(children=list(children))

    widgets = _FallbackWidgets()

_WIDGET_STYLE = {'description_width': '180px'}
_WIDGET_LAYOUT = widgets.Layout(width='100%')
_TEXT_LAYOUT = widgets.Layout(width='100%')
_MULTI_LAYOUT = widgets.Layout(width='100%', height='142px')
_CARD_LAYOUT = widgets.Layout(width='100%', padding='14px 16px 16px 16px')
_GRID_LAYOUT = widgets.Layout(
    width='100%',
    grid_template_columns='repeat(auto-fit, minmax(360px, 1fr))',
    grid_gap='12px 18px',
)

_PANEL_CSS = """
<style>
.deepmzyme-panel {
  --dmz-text: #0f172a;
  --dmz-muted: #64748b;
  --dmz-line: #d7dee8;
  --dmz-accent: #2563eb;
  --dmz-accent-soft: #dbeafe;
  font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  color: var(--dmz-text);
}
.deepmzyme-panel .widget-label,
.deepmzyme-panel label,
.deepmzyme-panel .widget-checkbox > label {
  color: var(--dmz-text) !important;
  font-weight: 650 !important;
}
.deepmzyme-panel input,
.deepmzyme-panel select,
.deepmzyme-panel textarea {
  background: #ffffff !important;
  color: #111827 !important;
  border: 1px solid #cbd5e1 !important;
  border-radius: 6px !important;
  min-height: 30px;
}
.deepmzyme-panel select[multiple] {
  padding: 4px !important;
  line-height: 1.32 !important;
}
.deepmzyme-panel select option:checked {
  color: #0f172a !important;
  background: linear-gradient(0deg, var(--dmz-accent-soft), var(--dmz-accent-soft)) !important;
  font-weight: 700 !important;
}
.deepmzyme-panel .widget-tab > .p-TabBar .p-TabBar-tab {
  background: #eef2f7;
  border: 1px solid var(--dmz-line);
  border-radius: 8px 8px 0 0;
  color: var(--dmz-text);
  font-weight: 650;
}
.deepmzyme-panel .widget-tab > .p-TabBar .p-TabBar-tab.p-mod-current {
  background: #ffffff;
  color: var(--dmz-accent);
  border-bottom-color: #ffffff;
}
.deepmzyme-panel .widget-tab > .widget-tab-contents {
  background: #ffffff;
  border: 1px solid var(--dmz-line);
  border-radius: 0 8px 8px 8px;
  padding: 12px;
}
.deepmzyme-panel details {
  background: #f8fafc;
  border: 1px solid #d7dee8;
  border-radius: 8px;
  padding: 9px 11px;
  color: #334155;
  font-size: 12px;
  line-height: 1.45;
}
.deepmzyme-panel summary {
  cursor: pointer;
  font-weight: 750;
  color: #1e3a8a;
}
</style>
"""

SELECTION_METRIC_OPTIONS = [
    '', 'train_loss', 'val_loss', 'val_metal_acc', 'val_ec_acc', 'val_ec_group_acc',
    'val_ec_group_balanced_acc', 'val_ec_group_macro_f1',
    'val_ec_group_level_1_acc', 'val_ec_group_level_1_balanced_acc', 'val_ec_group_level_1_macro_f1',
    'val_ec_group_level_2_acc', 'val_ec_group_level_2_balanced_acc', 'val_ec_group_level_2_macro_f1',
    'val_ec_group_level_3_acc', 'val_ec_group_level_3_balanced_acc', 'val_ec_group_level_3_macro_f1',
    'val_ec_group_level_4_acc', 'val_ec_group_level_4_balanced_acc', 'val_ec_group_level_4_macro_f1',
    'val_joint_balanced_acc', 'val_joint_macro_f1', 'val_metal_balanced_acc',
    'val_metal_min_recall', 'val_metal_mn_recall', 'val_metal_fe_recall', 'val_metal_class_viii_recall',
    'val_metal_collapsed4_acc', 'val_metal_collapsed4_balanced_acc', 'val_metal_collapsed4_macro_f1',
    'val_metal_collapsed4_mn_recall', 'val_ec_balanced_acc', 'val_metal_macro_f1', 'val_ec_macro_f1',
]


def _current_value(name, default):
    return globals().get(name, default)


def _as_list(value):
    if isinstance(value, (list, tuple, set)):
        return list(value)
    return [value]


def _selected_defaults(options, defaults):
    selected = [value for value in _as_list(defaults) if value in options]
    return tuple(selected or options[:1])


def _combo(description, options, value, *, placeholder='', tooltip=''):
    options = [str(option) for option in options]
    value = str(value).strip()
    if value and value not in options:
        options = [value] + options
    return widgets.Combobox(
        description=description,
        options=options,
        value=value if value in options or value else (options[0] if options else ''),
        ensure_option=False,
        placeholder=placeholder,
        style=_WIDGET_STYLE,
        layout=_WIDGET_LAYOUT,
        tooltip=tooltip,
    )


def _dropdown(description, options, value, *, tooltip=''):
    options = list(options)
    value = str(value).strip()
    if value not in options:
        value = options[0]
    return widgets.Dropdown(
        description=description,
        options=options,
        value=value,
        style=_WIDGET_STYLE,
        layout=_WIDGET_LAYOUT,
        tooltip=tooltip,
    )


def _multi_select(description, options, selected_values, *, tooltip=''):
    options = list(options)
    return widgets.SelectMultiple(
        description=description,
        options=options,
        value=_selected_defaults(options, selected_values),
        rows=min(max(len(options), 3), 8),
        style=_WIDGET_STYLE,
        layout=_MULTI_LAYOUT,
        tooltip=tooltip,
    )


def _int_text(description, value, *, tooltip=''):
    return widgets.IntText(description=description, value=int(value), style=_WIDGET_STYLE, layout=_WIDGET_LAYOUT, tooltip=tooltip)


def _float_text(description, value, *, tooltip=''):
    return widgets.FloatText(description=description, value=float(value), style=_WIDGET_STYLE, layout=_WIDGET_LAYOUT, tooltip=tooltip)


def _text(description, value='', placeholder='', *, tooltip=''):
    return widgets.Text(description=description, value=str(value), placeholder=placeholder, style=_WIDGET_STYLE, layout=_TEXT_LAYOUT, tooltip=tooltip)


def _csv_text(description, values, placeholder, *, tooltip=''):
    text = ','.join(str(value) for value in _as_list(values))
    return _text(description, text, placeholder, tooltip=tooltip)


def _checkbox(description, value, *, tooltip=''):
    return widgets.Checkbox(description=description, value=bool(value), indent=False, layout=_WIDGET_LAYOUT, tooltip=tooltip)


def _info(title, body):
    return widgets.HTML(value=f'<details><summary>Info: {title}</summary><div style="margin-top:7px;">{body}</div></details>')


def _section(title, subtitle, controls, *, columns=True):
    header = widgets.HTML(
        value=(
            f'<div style="margin:0 0 10px 0;">'
            f'<div style="font-size:16px;font-weight:750;color:#0f172a;">{title}</div>'
            f'<div style="font-size:12px;color:#64748b;margin-top:2px;">{subtitle}</div>'
            f'</div>'
        )
    )
    body = widgets.GridBox(controls, layout=_GRID_LAYOUT) if columns else widgets.VBox(controls)
    return widgets.VBox([header, body], layout=widgets.Layout(width='100%', padding='14px 16px 16px 16px'))


def _set_visible(widget, visible):
    layout = getattr(widget, 'layout', None)
    if layout is not None:
        layout.display = None if visible else 'none'


def _path_string(name, default):
    return str(_current_value(name, default))

_path_widgets = {
    'TRAIN_DIR': _text('Train structures', _path_string('train_dir', ''), 'path to train structures'),
    'TRAIN_CSV': _text('Train summary CSV', _path_string('train_csv', ''), 'site-level CSV'),
    'TEST_DIR': _text('Test structures', _path_string('test_dir', ''), 'path to held-out structures'),
    'TEST_CSV': _text('Test summary CSV', _path_string('test_csv', ''), 'site-level CSV'),
    'RUNS_DIR': _text('Runs output dir', _path_string('local_runs_dir', 'deepmzyme_outputs/runs'), 'run output root'),
    'EXTERNAL_FEATURES_ROOT_DIR': _text('External feature root', _current_value('EXTERNAL_FEATURES_ROOT_DIR', ''), 'blank = code default'),
    'EXTERNAL_FEATURE_SOURCE': _combo('External feature source', ['auto', 'updated', 'bluues_rosetta'], _current_value('EXTERNAL_FEATURE_SOURCE', 'auto')),
}

_run_widgets = {
    'EPOCHS': _int_text('Epochs', _current_value('EPOCHS', 50), tooltip='Maximum training epochs for each planned run.'),
    'VAL_FRACTION': _float_text('Validation fraction', _current_value('VAL_FRACTION', 0.15), tooltip='Fraction of training structures reserved for validation/model selection.'),
    'DEVICE': _combo('Device', ['cuda', 'cpu'], _current_value('DEVICE', 'cuda'), tooltip='Use cuda for real runs, cpu for short smoke checks.'),
    'RUN_NAME': _text('Run name', _current_value('RUN_NAME', ''), 'optional; only used when grid has one run'),
    'SPLIT_BY': _combo('Split by', ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id'], _current_value('SPLIT_BY', 'pdbid')),
    'N_FOLDS': _text('N folds', _current_value('N_FOLDS', ''), 'blank = no K-fold'),
    'FOLD_INDEX': _text('Fold index', _current_value('FOLD_INDEX', ''), 'blank = no K-fold'),
    'SELECTION_METRIC_OVERRIDE': _combo('Selection metric', SELECTION_METRIC_OPTIONS, _current_value('SELECTION_METRIC_OVERRIDE', ''), placeholder='blank = task default'),
    'LR_STEP_SIZE': _int_text('LR step size', _current_value('LR_STEP_SIZE', 10), tooltip='Used only when LR schedule is step.'),
    'LR_DECAY_GAMMA': _float_text('LR decay gamma', _current_value('LR_DECAY_GAMMA', 0.5), tooltip='Used only when LR schedule is step.'),
    'DETERMINISTIC': _checkbox('Deterministic mode', _current_value('DETERMINISTIC', False)),
    'SAVE_EPOCH_CHECKPOINTS': _checkbox('Save epoch checkpoints', _current_value('SAVE_EPOCH_CHECKPOINTS', False)),
    'RUN_HELD_OUT_TEST_EVAL': _checkbox('Run held-out test eval', _current_value('RUN_HELD_OUT_TEST_EVAL', True)),
    'ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG': _checkbox('Debug train-loss test eval', _current_value('ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG', False)),
}

_model_widgets = {
    'ESM_DIM': _int_text('ESM dim', _current_value('ESM_DIM', 960)),
    'NODE_RBF_SIGMA': _float_text('Node RBF sigma', _current_value('NODE_RBF_SIGMA', 0.75)),
    'EDGE_RBF_SIGMA': _float_text('Edge RBF sigma', _current_value('EDGE_RBF_SIGMA', 0.75)),
    'NODE_RBF_USE_RAW_DISTANCES': _checkbox('Raw node RBF distances', _current_value('NODE_RBF_USE_RAW_DISTANCES', False)),
    'DISABLE_ESM_BRANCH': _checkbox('Disable ESM branch', _current_value('DISABLE_ESM_BRANCH', False), tooltip='Advanced ablation; prefer Only-GVP for the standard graph-only baseline.'),
}

_esm_widgets = {
    'USE_ESM_EMBEDDINGS': _checkbox('Use ESM embeddings', _current_value('USE_ESM_EMBEDDINGS', True)),
    'ESM_EMBEDDINGS_DIR': _text('ESM embeddings dir', _current_value('ESM_EMBEDDINGS_DIR', DEFAULT_ESM_EMBEDDINGS_DIR)),
    'PREPARE_MISSING_ESM_EMBEDDINGS': _checkbox('Prepare missing ESM embeddings', _current_value('PREPARE_MISSING_ESM_EMBEDDINGS', False)),
    'ALLOW_MISSING_ESM_EMBEDDINGS': _checkbox('Allow missing ESM embeddings', _current_value('ALLOW_MISSING_ESM_EMBEDDINGS', False)),
    'ALLOW_MISSING_EXTERNAL_FEATURES': _checkbox('Allow missing external features', _current_value('ALLOW_MISSING_EXTERNAL_FEATURES', True)),
    # These widget globals are the actual defaults for early/hybrid ESM fusion presets.
    'EARLY_ESM_DIM': _int_text('Early ESM dim', _current_value('EARLY_ESM_DIM', 32)),
    'EARLY_ESM_DROPOUT': _float_text('Early ESM dropout', _current_value('EARLY_ESM_DROPOUT', 0.2)),
    'EARLY_ESM_RAW': _checkbox('Use raw early ESM', _current_value('EARLY_ESM_RAW', False)),
    'EARLY_ESM_SCOPE': _combo('Early ESM scope', ['all', 'first_shell', 'first_second_shell'], _current_value('EARLY_ESM_SCOPE', 'all')),
}

_ring_widgets = {
    'RING_EXE_PATH': _text('RING executable', _current_value('RING_EXE_PATH', DEFAULT_RING_EXE_PATH)),
    'RING_EDGE_OUTPUT_DIR': _text('RING output dir', _current_value('RING_EDGE_OUTPUT_DIR', DEFAULT_RING_EDGE_OUTPUT_DIR)),
    'PRECOMPUTED_RING_EDGES_DIR': _text('Precomputed RING dir', _current_value('PRECOMPUTED_RING_EDGES_DIR', DEFAULT_PRECOMPUTED_RING_EDGES_DIR)),
    'REQUIRE_RING_EDGES': _checkbox('Require RING edges', _current_value('REQUIRE_RING_EDGES', False)),
    'PREPARE_MISSING_RING_EDGES': _checkbox('Prepare missing RING edges', _current_value('PREPARE_MISSING_RING_EDGES', False)),
}

_loss_widgets = {
    'BALANCE_METAL_SITE_SYMBOLS': _checkbox('Balance metal site symbols', _current_value('BALANCE_METAL_SITE_SYMBOLS', False)),
    'REQUIRE_ALL_TASK_CLASSES': _checkbox('Require all task classes', _current_value('REQUIRE_ALL_TASK_CLASSES', False)),
    'METAL_LOSS_FUNCTION': _combo('Metal loss function', ['cross_entropy', 'focal'], _current_value('METAL_LOSS_FUNCTION', 'cross_entropy')),
    'METAL_FOCAL_GAMMA': _float_text('Metal focal gamma', _current_value('METAL_FOCAL_GAMMA', 2.0)),
    'METAL_LABEL_SMOOTHING': _float_text('Metal label smoothing', _current_value('METAL_LABEL_SMOOTHING', 0.0)),
    'METAL_LOSS_WEIGHT': _float_text('Metal loss weight', _current_value('METAL_LOSS_WEIGHT', 1.0)),
    'EC_LOSS_WEIGHT': _float_text('EC loss weight', _current_value('EC_LOSS_WEIGHT', 1.0)),
    'MN_LOSS_MULTIPLIER': _float_text('Mn loss multiplier', _current_value('MN_LOSS_MULTIPLIER', 1.0)),
    'CU_LOSS_MULTIPLIER': _float_text('Cu loss multiplier', _current_value('CU_LOSS_MULTIPLIER', 1.0)),
    'ZN_LOSS_MULTIPLIER': _float_text('Zn loss multiplier', _current_value('ZN_LOSS_MULTIPLIER', 1.0)),
    'FE_LOSS_MULTIPLIER': _float_text('Fe loss multiplier', _current_value('FE_LOSS_MULTIPLIER', 1.0)),
    'CO_LOSS_MULTIPLIER': _float_text('Co loss multiplier', _current_value('CO_LOSS_MULTIPLIER', 1.0)),
    'NI_LOSS_MULTIPLIER': _float_text('Ni loss multiplier', _current_value('NI_LOSS_MULTIPLIER', 1.0)),
    'CLASS_VIII_LOSS_MULTIPLIER': _float_text('Class VIII loss multiplier', _current_value('CLASS_VIII_LOSS_MULTIPLIER', 1.0)),
    'EC_GROUP_WEIGHTING': _combo('EC group weighting', ['structure_id', 'pdbid_chain', 'pdbid', 'none'], _current_value('EC_GROUP_WEIGHTING', 'structure_id')),
    'EC_CONTRASTIVE_TEMPERATURE': _float_text('EC contrastive temp', _current_value('EC_CONTRASTIVE_TEMPERATURE', 0.1)),
    'UNSUPPORTED_METAL_POLICY': _combo('Unsupported metal policy', ['error', 'skip'], _current_value('UNSUPPORTED_METAL_POLICY', 'error')),
    'INVALID_STRUCTURE_POLICY': _combo('Invalid structure policy', ['skip', 'error'], _current_value('INVALID_STRUCTURE_POLICY', 'skip')),
}

_multi_select_widgets = {
    'TASK_MODES': _multi_select('Task modes', list(task_map), _current_value('TASK_MODES', ['metal_6_class'])),
    'MODEL_PRESETS': _multi_select('Model presets', list(preset_map), _current_value('MODEL_PRESETS', ['Only-GVP'])),
    'RING_EDGES_MODES': _multi_select('RING modes', list(RING_EDGE_MODE_LABEL_TO_VALUE), [ring_edges_mode_label(mode) for mode in _current_value('RING_EDGES_MODES', ['radius_only'])]),
    'NODE_FEATURES_OMIT': widgets.SelectMultiple(
        description='Omit node features',
        options=CONSERVATIVE_NODE_FEATURES,
        value=tuple(_current_value('NODE_FEATURES_OMIT', [])),
        rows=min(len(CONSERVATIVE_NODE_FEATURES), 8),
        style=_WIDGET_STYLE,
        layout=widgets.Layout(width='100%', height='300px'),
        tooltip='Keeps --node-feature-set conservative and passes selected omissions separately. Default: nothing selected = use all features.',
    ),
    'EC_LABEL_DEPTHS': _multi_select('EC label depths', [1, 2, 3, 4], _current_value('EC_LABEL_DEPTHS', [1])),
    'CROSS_ATTENTION_LAYERS': _multi_select('Cross-attn layers', [1, 2, 3], _current_value('CROSS_ATTENTION_LAYERS', [1])),
    'CROSS_ATTENTION_HEADS': _multi_select('Cross-attn heads', [2, 4, 8], _current_value('CROSS_ATTENTION_HEADS', [4])),
    'LR_SCHEDULES': _multi_select('LR schedules', ['fixed', 'cosine', 'step'], _current_value('LR_SCHEDULES', ['fixed'])),
}

_multi_custom_widgets = {
    'TASK_MODES': _text('Custom task modes CSV', '', 'must match supported CLI choices'),
    'RING_EDGES_MODES': _text('Custom RING modes CSV', '', 'radius_only,...'),
    'EC_LABEL_DEPTHS': _text('Custom EC depths CSV', '', '1,2,3,4'),
    'CROSS_ATTENTION_LAYERS': _text('Custom cross layers CSV', '', '1,2,3'),
    'CROSS_ATTENTION_HEADS': _text('Custom cross heads CSV', '', '2,4,8'),
}

_grid_csv_widgets = {
    'LEARNING_RATES': _csv_text('Learning rates CSV', _current_value('LEARNING_RATES', [3e-5]), '3e-5,1e-4'),
    'WEIGHT_DECAYS': _csv_text('Weight decays CSV', _current_value('WEIGHT_DECAYS', [1e-4]), '1e-4,0'),
    'BATCH_SIZES': _csv_text('Batch sizes CSV', _current_value('BATCH_SIZES', [8]), '4,8,16'),
    'SEEDS': _csv_text('Seeds CSV', _current_value('SEEDS', [42]), '42,7,123'),
    'EC_CONTRASTIVE_WEIGHTS': _csv_text('EC contrast CSV', _current_value('EC_CONTRASTIVE_WEIGHTS', [0.0]), '0,0.1,0.5'),
    'GVP_LAYER_VALUES': _csv_text('GVP layers CSV', _current_value('GVP_LAYER_VALUES', [4]), '4,6'),
    'HIDDEN_S_VALUES': _csv_text('Hidden_s CSV', _current_value('HIDDEN_S_VALUES', [128]), '128,192'),
    'HIDDEN_V_VALUES': _csv_text('Hidden_v CSV', _current_value('HIDDEN_V_VALUES', [16]), '16,32'),
    'EDGE_HIDDEN_VALUES': _csv_text('Edge hidden CSV', _current_value('EDGE_HIDDEN_VALUES', [64]), '64,96'),
    'EDGE_RADIUS_VALUES': _csv_text('Edge radius CSV', _current_value('EDGE_RADIUS_VALUES', [8.0]), '8,10,12'),
    'ESM_FUSION_DIM_VALUES': _csv_text('ESM fusion dim CSV', _current_value('ESM_FUSION_DIM_VALUES', [128]), '128,256'),
    'HEAD_MLP_LAYER_VALUES': _csv_text('Head MLP layers CSV', _current_value('HEAD_MLP_LAYER_VALUES', [2]), '2,3'),
}

_sweep_controls = {
    'MAX_SWEEP_RUNS': _int_text('Max grid runs', _current_value('MAX_SWEEP_RUNS', 24)),
    'STOP_ON_FIRST_FAILURE': _checkbox('Stop on first failure', _current_value('STOP_ON_FIRST_FAILURE', True)),
    'SKIP_EXISTING_RUNS': _checkbox('Skip existing runs', _current_value('SKIP_EXISTING_RUNS', True)),
}

_execution_widgets = {
    'COPY_OUTPUTS_TO_DRIVE': _checkbox('Copy outputs to Drive', _current_value('COPY_OUTPUTS_TO_DRIVE', RUN_MODE == 'colab' and globals().get('use_drive_paths', False))),
}

intro = widgets.HTML(
    value=(
        '<div style="background:#0f172a;color:#f8fafc;border-radius:10px;padding:16px 18px;">'
        '<div style="font-size:20px;font-weight:800;letter-spacing:0;">DeepMzyme Training Configuration</div>'
        '<div style="font-size:13px;color:#cbd5e1;margin-top:5px;line-height:1.45;">'
        'Default settings use Only-GVP and radius-only edges so the first run is independent of unavailable ESM/RING files.'
        '</div></div>'
    )
)

_lr_step_section = widgets.VBox(
    [_run_widgets['LR_STEP_SIZE'], _run_widgets['LR_DECAY_GAMMA']],
    layout=widgets.Layout(width='100%'),
)
_cross_run_widgets = {
    'CROSS_ATTENTION_DROPOUT': _float_text('Cross-attn dropout', _current_value('CROSS_ATTENTION_DROPOUT', 0.1)),
    'CROSS_ATTENTION_NEIGHBORHOOD': _combo('Cross-attn neighborhood', ['all', 'first_shell', 'first_second_shell'], _current_value('CROSS_ATTENTION_NEIGHBORHOOD', 'all')),
    'CROSS_ATTENTION_BIDIRECTIONAL': _checkbox('Cross-attn bidirectional', _current_value('CROSS_ATTENTION_BIDIRECTIONAL', False)),
}
_run_cross_attention_section = widgets.VBox(
    list(_cross_run_widgets.values()),
    layout=widgets.Layout(width='100%'),
)

path_tab = _section(
    'Data paths and output paths',
    'Training/test inputs, external feature root, and run output directory used by generated CLI commands.',
    list(_path_widgets.values()) + [
        _info('path safety', 'In Colab, paths under /home/... point to a different machine and will trigger warnings. Use /content/... or mounted Drive paths there.'),
    ],
)

shared_tab = _section(
    'Training, validation, and reporting',
    'Runtime, split, scheduler, checkpoint selection, reproducibility, and held-out test reporting.',
    list(_run_widgets.values()) + [
        _lr_step_section,
        _run_cross_attention_section,
        _loss_widgets['INVALID_STRUCTURE_POLICY'],
        _info('validation and test use', 'Validation metrics select checkpoints and compare model variants. Held-out test evaluation is for final reporting of the selected checkpoint only.'),
    ],
)

esm_tab = _section(
    'ESM embeddings and early fusion',
    'Controls for precomputed/generated ESM embeddings plus early residue-level ESM options.',
    list(_esm_widgets.values()) + [
        _info('ESM requirements', 'Only-ESM and graph+ESM presets require precomputed ESMC residue embeddings unless missing embeddings are explicitly allowed or generated.'),
        _info('raw early ESM', 'Raw early ESM is a high-dimensional ablation. Prefer the compressed early ESM dim unless deliberately testing this path.'),
    ],
)

ring_tab = _section(
    'RING edge settings',
    'Use radius-only first; enable precomputed or generated RING edges for focused ablations.',
    list(_ring_widgets.values()) + [
        _info('RING modes', 'Radius-only uses geometric residue-neighbor edges. Existing RING files read PRECOMPUTED_RING_EDGES_DIR. Generated RING writes to RING_EDGE_OUTPUT_DIR.'),
    ],
)

_task_controls = []
for name in ['TASK_MODES', 'MODEL_PRESETS', 'LR_SCHEDULES']:
    _task_controls.append(_multi_select_widgets[name])
    if name not in {'MODEL_PRESETS', 'LR_SCHEDULES'}:
        _task_controls.append(_multi_custom_widgets[name])
_task_section = _section(
    'Task selection, model architecture, and fixed choices',
    'Select one value for a single run or multiple values for a sweep. Node features stay on the conservative preset, with optional omissions.',
    _task_controls,
)

_metal_section = _section(
    'Metal-specific options',
    'Metal sampler, metal loss, per-class loss multipliers, and label/structure policies.',
    [
        _loss_widgets['BALANCE_METAL_SITE_SYMBOLS'], _loss_widgets['REQUIRE_ALL_TASK_CLASSES'],
        _loss_widgets['METAL_LOSS_FUNCTION'], _loss_widgets['METAL_FOCAL_GAMMA'], _loss_widgets['METAL_LABEL_SMOOTHING'],
        _loss_widgets['METAL_LOSS_WEIGHT'], _loss_widgets['MN_LOSS_MULTIPLIER'], _loss_widgets['CU_LOSS_MULTIPLIER'],
        _loss_widgets['ZN_LOSS_MULTIPLIER'], _loss_widgets['FE_LOSS_MULTIPLIER'], _loss_widgets['CO_LOSS_MULTIPLIER'],
        _loss_widgets['NI_LOSS_MULTIPLIER'], _loss_widgets['CLASS_VIII_LOSS_MULTIPLIER'],
        _loss_widgets['UNSUPPORTED_METAL_POLICY'],
    ],
)

_ec_controls = [_multi_select_widgets['EC_LABEL_DEPTHS'], _multi_custom_widgets['EC_LABEL_DEPTHS'], _grid_csv_widgets['EC_CONTRASTIVE_WEIGHTS'], _loss_widgets['EC_LOSS_WEIGHT'], _loss_widgets['EC_GROUP_WEIGHTING'], _loss_widgets['EC_CONTRASTIVE_TEMPERATURE']]
_ec_section = _section(
    'EC/function-specific controls',
    'EC label depth, group weighting, task loss weight, and optional contrastive loss.',
    _ec_controls + [
        _info('EC group weighting', 'The default structure_id weighting prevents multiple separated pockets from one structure from over-counting the EC objective.'),
    ],
)

_model_section = _section(
    'Model architecture and GVP settings',
    'Hidden dimensions, GVP/GNN depth, edge radius, classifier head size, and encoder RBF settings.',
    list(_model_widgets.values()) + [
        _grid_csv_widgets['GVP_LAYER_VALUES'], _grid_csv_widgets['HIDDEN_S_VALUES'], _grid_csv_widgets['HIDDEN_V_VALUES'],
        _grid_csv_widgets['EDGE_HIDDEN_VALUES'], _grid_csv_widgets['EDGE_RADIUS_VALUES'], _grid_csv_widgets['ESM_FUSION_DIM_VALUES'],
        _grid_csv_widgets['HEAD_MLP_LAYER_VALUES'],
        _multi_select_widgets['NODE_FEATURES_OMIT'],
        _info('capacity controls', 'gvp_layers controls message-passing depth. hidden_s/hidden_v are scalar/vector channels. edge_hidden is the edge encoder width. head_mlp_layers controls classifier depth.'),
        _info('GPU memory guidance', 'Colab T4 has about 15 GB VRAM. Safe baseline: gvp_layers=4, hidden_s=128, hidden_v=16, edge_hidden=64, batch_size=8. If using hidden_s=192, hidden_v=32, or gvp_layers >= 6, reduce batch_size to 4 or smaller. Only-ESM uses much less VRAM than GVP variants.'),
    ],
)

_cross_controls = []
for name in ['CROSS_ATTENTION_LAYERS', 'CROSS_ATTENTION_HEADS']:
    _cross_controls.extend([_multi_select_widgets[name], _multi_custom_widgets[name]])
_cross_attention_section = _section(
    'Fusion and cross-attention settings',
    'Cross-modal attention layer/head sweeps are active only for the GVP + cross-modal attention preset.',
    _cross_controls + [
        _info('cross-modal attention', 'Graph/pocket states attend to ESM residue states. Keep this advanced fusion off until simpler baselines are stable.'),
    ],
)

_numeric_section = _section(
    'Training hyperparameter sweeps',
    'Comma-separated numeric lists create the Cartesian product used by the command builder.',
    [_grid_csv_widgets['LEARNING_RATES'], _grid_csv_widgets['WEIGHT_DECAYS'], _grid_csv_widgets['BATCH_SIZES'], _grid_csv_widgets['SEEDS']],
)

_ring_mode_section = _section(
    'RING sweep choices',
    'Select RING edge variants to include in the grid.',
    [_multi_select_widgets['RING_EDGES_MODES'], _multi_custom_widgets['RING_EDGES_MODES']],
)

_sweep_section = _section(
    'Sweep guards',
    'Caps and failure behavior for long-running grid execution.',
    list(_sweep_controls.values()),
    columns=False,
)

grid_tab = widgets.VBox([
    _task_section,
    _model_section,
    _numeric_section,
    _ec_section,
    _metal_section,
    _cross_attention_section,
    _ring_mode_section,
    _sweep_section,
], layout=widgets.Layout(width='100%'))

execution_tab = _section(
    'Output copy controls',
    'Training behavior is inferred automatically: one planned configuration runs once; more than one planned configuration runs as a sweep.',
    list(_execution_widgets.values()) + [
        _info('inferred training behavior', 'Select one value per configurable field for one run. Select multiple values in any field to create multiple planned runs; the training cell will run them as a sweep.'),
    ],
    columns=False,
)


def _has_ec_task():
    return any(task in {'ec_prediction', 'joint_metal_ec'} for task in _multi_select_widgets['TASK_MODES'].value)


def _has_metal_task():
    return any(task in {'metal_6_class', 'metal_collapsed4_metric', 'joint_metal_ec'} for task in _multi_select_widgets['TASK_MODES'].value)


def _has_cross_attention_model():
    return 'GVP + cross-modal attention' in _multi_select_widgets['MODEL_PRESETS'].value


def _has_esm_model():
    return any(preset_map.get(p, {}).get('uses_esm', False)
               for p in _multi_select_widgets['MODEL_PRESETS'].value)


def _refresh_contextual_sections(_change=None):
    _set_visible(_metal_section, _has_metal_task())
    _set_visible(_ec_section, _has_ec_task())
    _set_visible(_cross_attention_section, _has_cross_attention_model())
    _set_visible(_run_cross_attention_section, _has_cross_attention_model())
    _set_visible(_model_widgets['DISABLE_ESM_BRANCH'], _has_esm_model())


def _refresh_lr_step_section(_change=None):
    _set_visible(_lr_step_section, 'step' in _multi_select_widgets['LR_SCHEDULES'].value)


if hasattr(_multi_select_widgets['TASK_MODES'], 'observe'):
    _multi_select_widgets['TASK_MODES'].observe(_refresh_contextual_sections, names='value')
if hasattr(_multi_select_widgets['MODEL_PRESETS'], 'observe'):
    _multi_select_widgets['MODEL_PRESETS'].observe(_refresh_contextual_sections, names='value')
if hasattr(_multi_select_widgets['LR_SCHEDULES'], 'observe'):
    _multi_select_widgets['LR_SCHEDULES'].observe(_refresh_lr_step_section, names='value')
_refresh_contextual_sections()
_refresh_lr_step_section()

_panel = widgets.Tab(children=[path_tab, shared_tab, esm_tab, ring_tab, grid_tab, execution_tab], layout=widgets.Layout(width='100%'))
for index, title in enumerate(['Paths', 'Run', 'ESM', 'RING', 'Grid', 'Output']):
    _panel.set_title(index, title)

_dashboard = widgets.VBox([intro, _panel], layout=widgets.Layout(width='100%', max_width='1180px'))
if hasattr(_dashboard, 'add_class'):
    _dashboard.add_class('deepmzyme-panel')

if _WIDGET_PANEL_AVAILABLE:
    display(HTML(_PANEL_CSS))
    display(_dashboard)
else:
    print('ipywidgets is not installed; interactive panel skipped.')
    print('Existing #@param/default values were captured, so the next configuration-read cell can still run.')


In [92]:
#@title Read widget values into final configuration variables
import json
from pathlib import Path

required_widget_groups = ['_path_widgets', '_run_widgets', '_model_widgets', '_esm_widgets', '_ring_widgets', '_loss_widgets', '_multi_select_widgets', '_multi_custom_widgets', '_grid_csv_widgets', '_execution_widgets']
missing_groups = [name for name in required_widget_groups if name not in globals()]
if missing_groups:
    raise RuntimeError("Run the 'Interactive configuration panel' cell first, or skip both widget cells and use the existing #@param configuration cells. Missing: " + ', '.join(missing_groups))


def _csv_tokens(text):
    return [token.strip() for token in str(text or '').split(',') if token.strip()]


def _parse_csv_numbers(text, converter, label):
    tokens = _csv_tokens(text)
    if not tokens:
        raise ValueError(f'{label} must contain at least one comma-separated value.')
    parsed = []
    for value in tokens:
        try:
            parsed.append(converter(value))
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Could not parse {label} value {value!r} from comma-separated values: {text!r}') from exc
    return parsed


def parse_csv_ints(text, label):
    return _parse_csv_numbers(text, int, label)


def parse_csv_floats(text, label):
    return _parse_csv_numbers(text, float, label)


def _text_value(widgets_by_name, name):
    return str(widgets_by_name[name].value).strip()


def _choice_value(widgets_by_name, name):
    return str(widgets_by_name[name].value).strip()


def _selected_values(name):
    selected = list(_multi_select_widgets[name].value)
    custom_widget = _multi_custom_widgets.get(name)
    custom = [] if custom_widget is None or name == 'MODEL_PRESETS' else _csv_tokens(custom_widget.value)
    values = selected + custom
    deduped = []
    for value in values:
        if value not in deduped:
            deduped.append(value)
    return deduped


def _validate_choice(name, value, options, *, allow_blank=False):
    if allow_blank and value == '':
        return
    if value not in options:
        valid = ', '.join(str(option) for option in options)
        raise ValueError(f'{name}={value!r} is not valid. Choose one of: {valid}')


def _validate_choices(name, values, options):
    invalid = [value for value in values if value not in options]
    if invalid:
        valid = ', '.join(str(option) for option in options)
        raise ValueError(f'{name} contains invalid value(s) {invalid!r}. Choose from: {valid}')
    if not values:
        raise ValueError(f'{name} must contain at least one value.')


def _optional_int(text, label):
    text = str(text).strip()
    if not text:
        return None
    try:
        return int(text)
    except ValueError as exc:
        raise ValueError(f'{label} must be blank or an integer, got {text!r}.') from exc


def _path_from_widget(value, *, base=repo_dir):
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path
    return (Path(base) / path).resolve()


TASK_MODE_OPTIONS = list(task_map)
MODEL_PRESET_OPTIONS = list(preset_map)
DEVICE_OPTIONS = ['cuda', 'cpu']
SPLIT_BY_OPTIONS = ['pdbid', 'pdbid_chain', 'structure_id', 'pocket_id']
LR_SCHEDULE_OPTIONS = ['fixed', 'cosine', 'step']
CROSS_ATTENTION_NEIGHBORHOOD_OPTIONS = ['all', 'first_shell', 'first_second_shell']
EARLY_ESM_SCOPE_OPTIONS = ['all', 'first_shell', 'first_second_shell']
EC_GROUP_WEIGHTING_OPTIONS = ['none', 'structure_id', 'pdbid_chain', 'pdbid']
METAL_LOSS_FUNCTION_OPTIONS = ['cross_entropy', 'focal']
EXTERNAL_FEATURE_SOURCE_OPTIONS = ['auto', 'bluues_rosetta', 'updated']
UNSUPPORTED_METAL_POLICY_OPTIONS = ['error', 'skip']
INVALID_STRUCTURE_POLICY_OPTIONS = ['error', 'skip']
EC_LABEL_DEPTH_OPTIONS = [1, 2, 3, 4]
CROSS_ATTENTION_LAYER_OPTIONS = [1, 2, 3]
CROSS_ATTENTION_HEAD_OPTIONS = [2, 4, 8]

train_dir = _path_from_widget(_text_value(_path_widgets, 'TRAIN_DIR'))
train_csv = _path_from_widget(_text_value(_path_widgets, 'TRAIN_CSV'))
test_dir = _path_from_widget(_text_value(_path_widgets, 'TEST_DIR'))
test_csv = _path_from_widget(_text_value(_path_widgets, 'TEST_CSV'))
local_runs_dir = _path_from_widget(_text_value(_path_widgets, 'RUNS_DIR'))
EXTERNAL_FEATURES_ROOT_DIR = _text_value(_path_widgets, 'EXTERNAL_FEATURES_ROOT_DIR')
EXTERNAL_FEATURE_SOURCE = _choice_value(_path_widgets, 'EXTERNAL_FEATURE_SOURCE')

EPOCHS = int(_run_widgets['EPOCHS'].value)
VAL_FRACTION = float(_run_widgets['VAL_FRACTION'].value)
DEVICE = _choice_value(_run_widgets, 'DEVICE')
RUN_NAME = _text_value(_run_widgets, 'RUN_NAME')
SPLIT_BY = _choice_value(_run_widgets, 'SPLIT_BY')
N_FOLDS = _optional_int(_run_widgets['N_FOLDS'].value, 'N_FOLDS')
FOLD_INDEX = _optional_int(_run_widgets['FOLD_INDEX'].value, 'FOLD_INDEX')
SELECTION_METRIC_OVERRIDE = _choice_value(_run_widgets, 'SELECTION_METRIC_OVERRIDE')
LR_STEP_SIZE = int(_run_widgets['LR_STEP_SIZE'].value)
LR_DECAY_GAMMA = float(_run_widgets['LR_DECAY_GAMMA'].value)
DETERMINISTIC = bool(_run_widgets['DETERMINISTIC'].value)
SAVE_EPOCH_CHECKPOINTS = bool(_run_widgets['SAVE_EPOCH_CHECKPOINTS'].value)
RUN_HELD_OUT_TEST_EVAL = bool(_run_widgets['RUN_HELD_OUT_TEST_EVAL'].value)
ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG = bool(_run_widgets['ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG'].value)
CROSS_ATTENTION_DROPOUT = float(_cross_run_widgets['CROSS_ATTENTION_DROPOUT'].value)
CROSS_ATTENTION_NEIGHBORHOOD = _choice_value(_cross_run_widgets, 'CROSS_ATTENTION_NEIGHBORHOOD')
CROSS_ATTENTION_BIDIRECTIONAL = bool(_cross_run_widgets['CROSS_ATTENTION_BIDIRECTIONAL'].value)

ESM_DIM = int(_model_widgets['ESM_DIM'].value)
NODE_RBF_SIGMA = float(_model_widgets['NODE_RBF_SIGMA'].value)
EDGE_RBF_SIGMA = float(_model_widgets['EDGE_RBF_SIGMA'].value)
NODE_RBF_USE_RAW_DISTANCES = bool(_model_widgets['NODE_RBF_USE_RAW_DISTANCES'].value)
DISABLE_ESM_BRANCH = bool(_model_widgets['DISABLE_ESM_BRANCH'].value)

USE_ESM_EMBEDDINGS = bool(_esm_widgets['USE_ESM_EMBEDDINGS'].value)
ESM_EMBEDDINGS_DIR = _text_value(_esm_widgets, 'ESM_EMBEDDINGS_DIR')
PREPARE_MISSING_ESM_EMBEDDINGS = bool(_esm_widgets['PREPARE_MISSING_ESM_EMBEDDINGS'].value)
ALLOW_MISSING_ESM_EMBEDDINGS = bool(_esm_widgets['ALLOW_MISSING_ESM_EMBEDDINGS'].value)
ALLOW_MISSING_EXTERNAL_FEATURES = bool(_esm_widgets['ALLOW_MISSING_EXTERNAL_FEATURES'].value)
EARLY_ESM_DIM = int(_esm_widgets['EARLY_ESM_DIM'].value)
EARLY_ESM_DROPOUT = float(_esm_widgets['EARLY_ESM_DROPOUT'].value)
EARLY_ESM_RAW = bool(_esm_widgets['EARLY_ESM_RAW'].value)
EARLY_ESM_SCOPE = _choice_value(_esm_widgets, 'EARLY_ESM_SCOPE')

RING_EXE_PATH = _text_value(_ring_widgets, 'RING_EXE_PATH')
RING_EDGE_OUTPUT_DIR = _text_value(_ring_widgets, 'RING_EDGE_OUTPUT_DIR')
PRECOMPUTED_RING_EDGES_DIR = _text_value(_ring_widgets, 'PRECOMPUTED_RING_EDGES_DIR')
REQUIRE_RING_EDGES = bool(_ring_widgets['REQUIRE_RING_EDGES'].value)
PREPARE_MISSING_RING_EDGES = bool(_ring_widgets['PREPARE_MISSING_RING_EDGES'].value)

BALANCE_METAL_SITE_SYMBOLS = bool(_loss_widgets['BALANCE_METAL_SITE_SYMBOLS'].value)
REQUIRE_ALL_TASK_CLASSES = bool(_loss_widgets['REQUIRE_ALL_TASK_CLASSES'].value)
METAL_LOSS_FUNCTION = _choice_value(_loss_widgets, 'METAL_LOSS_FUNCTION')
METAL_FOCAL_GAMMA = float(_loss_widgets['METAL_FOCAL_GAMMA'].value)
METAL_LABEL_SMOOTHING = float(_loss_widgets['METAL_LABEL_SMOOTHING'].value)
METAL_LOSS_WEIGHT = float(_loss_widgets['METAL_LOSS_WEIGHT'].value)
EC_LOSS_WEIGHT = float(_loss_widgets['EC_LOSS_WEIGHT'].value)
MN_LOSS_MULTIPLIER = float(_loss_widgets['MN_LOSS_MULTIPLIER'].value)
CU_LOSS_MULTIPLIER = float(_loss_widgets['CU_LOSS_MULTIPLIER'].value)
ZN_LOSS_MULTIPLIER = float(_loss_widgets['ZN_LOSS_MULTIPLIER'].value)
FE_LOSS_MULTIPLIER = float(_loss_widgets['FE_LOSS_MULTIPLIER'].value)
CO_LOSS_MULTIPLIER = float(_loss_widgets['CO_LOSS_MULTIPLIER'].value)
NI_LOSS_MULTIPLIER = float(_loss_widgets['NI_LOSS_MULTIPLIER'].value)
CLASS_VIII_LOSS_MULTIPLIER = float(_loss_widgets['CLASS_VIII_LOSS_MULTIPLIER'].value)
EC_GROUP_WEIGHTING = _choice_value(_loss_widgets, 'EC_GROUP_WEIGHTING')
EC_CONTRASTIVE_TEMPERATURE = float(_loss_widgets['EC_CONTRASTIVE_TEMPERATURE'].value)
UNSUPPORTED_METAL_POLICY = _choice_value(_loss_widgets, 'UNSUPPORTED_METAL_POLICY')
INVALID_STRUCTURE_POLICY = _choice_value(_loss_widgets, 'INVALID_STRUCTURE_POLICY')

TASK_MODES = _selected_values('TASK_MODES')
MODEL_PRESETS = _selected_values('MODEL_PRESETS')
RING_EDGES_MODES = [normalize_ring_edges_mode(mode) for mode in _selected_values('RING_EDGES_MODES')]
NODE_FEATURE_SET = 'conservative'
NODE_FEATURE_SETS = [NODE_FEATURE_SET]
NODE_FEATURES_OMIT = list(_multi_select_widgets['NODE_FEATURES_OMIT'].value)
EC_LABEL_DEPTHS = [int(value) for value in _selected_values('EC_LABEL_DEPTHS')]
CROSS_ATTENTION_LAYERS = [int(value) for value in _selected_values('CROSS_ATTENTION_LAYERS')]
CROSS_ATTENTION_HEADS = [int(value) for value in _selected_values('CROSS_ATTENTION_HEADS')]
LR_SCHEDULES = _selected_values('LR_SCHEDULES')

LEARNING_RATES = parse_csv_floats(_grid_csv_widgets['LEARNING_RATES'].value, 'LEARNING_RATES')
WEIGHT_DECAYS = parse_csv_floats(_grid_csv_widgets['WEIGHT_DECAYS'].value, 'WEIGHT_DECAYS')
BATCH_SIZES = parse_csv_ints(_grid_csv_widgets['BATCH_SIZES'].value, 'BATCH_SIZES')
SEEDS = parse_csv_ints(_grid_csv_widgets['SEEDS'].value, 'SEEDS')
EC_CONTRASTIVE_WEIGHTS = parse_csv_floats(_grid_csv_widgets['EC_CONTRASTIVE_WEIGHTS'].value, 'EC_CONTRASTIVE_WEIGHTS')
GVP_LAYER_VALUES = parse_csv_ints(_grid_csv_widgets['GVP_LAYER_VALUES'].value, 'GVP_LAYER_VALUES')
HIDDEN_S_VALUES = parse_csv_ints(_grid_csv_widgets['HIDDEN_S_VALUES'].value, 'HIDDEN_S_VALUES')
HIDDEN_V_VALUES = parse_csv_ints(_grid_csv_widgets['HIDDEN_V_VALUES'].value, 'HIDDEN_V_VALUES')
EDGE_HIDDEN_VALUES = parse_csv_ints(_grid_csv_widgets['EDGE_HIDDEN_VALUES'].value, 'EDGE_HIDDEN_VALUES')
EDGE_RADIUS_VALUES = parse_csv_floats(_grid_csv_widgets['EDGE_RADIUS_VALUES'].value, 'EDGE_RADIUS_VALUES')
ESM_FUSION_DIM_VALUES = parse_csv_ints(_grid_csv_widgets['ESM_FUSION_DIM_VALUES'].value, 'ESM_FUSION_DIM_VALUES')
HEAD_MLP_LAYER_VALUES = parse_csv_ints(_grid_csv_widgets['HEAD_MLP_LAYER_VALUES'].value, 'HEAD_MLP_LAYER_VALUES')

MAX_SWEEP_RUNS = int(_sweep_controls['MAX_SWEEP_RUNS'].value)
STOP_ON_FIRST_FAILURE = bool(_sweep_controls['STOP_ON_FIRST_FAILURE'].value)
SKIP_EXISTING_RUNS = bool(_sweep_controls['SKIP_EXISTING_RUNS'].value)

COPY_OUTPUTS_TO_DRIVE = bool(_execution_widgets['COPY_OUTPUTS_TO_DRIVE'].value)

_validate_choice('DEVICE', DEVICE, DEVICE_OPTIONS)
_validate_choice('SPLIT_BY', SPLIT_BY, SPLIT_BY_OPTIONS)
_validate_choice('CROSS_ATTENTION_NEIGHBORHOOD', CROSS_ATTENTION_NEIGHBORHOOD, CROSS_ATTENTION_NEIGHBORHOOD_OPTIONS)
_validate_choice('EARLY_ESM_SCOPE', EARLY_ESM_SCOPE, EARLY_ESM_SCOPE_OPTIONS)
_validate_choice('EC_GROUP_WEIGHTING', EC_GROUP_WEIGHTING, EC_GROUP_WEIGHTING_OPTIONS)
_validate_choice('METAL_LOSS_FUNCTION', METAL_LOSS_FUNCTION, METAL_LOSS_FUNCTION_OPTIONS)
_validate_choice('EXTERNAL_FEATURE_SOURCE', EXTERNAL_FEATURE_SOURCE, EXTERNAL_FEATURE_SOURCE_OPTIONS)
_validate_choice('UNSUPPORTED_METAL_POLICY', UNSUPPORTED_METAL_POLICY, UNSUPPORTED_METAL_POLICY_OPTIONS)
_validate_choice('INVALID_STRUCTURE_POLICY', INVALID_STRUCTURE_POLICY, INVALID_STRUCTURE_POLICY_OPTIONS)
_validate_choice('SELECTION_METRIC_OVERRIDE', SELECTION_METRIC_OVERRIDE, SELECTION_METRIC_OPTIONS, allow_blank=True)
_validate_choices('TASK_MODES', TASK_MODES, TASK_MODE_OPTIONS)
_validate_choices('MODEL_PRESETS', MODEL_PRESETS, MODEL_PRESET_OPTIONS)
_validate_choices('NODE_FEATURES_OMIT', NODE_FEATURES_OMIT, CONSERVATIVE_NODE_FEATURES)
_validate_choices('LR_SCHEDULES', LR_SCHEDULES, LR_SCHEDULE_OPTIONS)
_validate_choices('EC_LABEL_DEPTHS', EC_LABEL_DEPTHS, EC_LABEL_DEPTH_OPTIONS)
_validate_choices('CROSS_ATTENTION_LAYERS', CROSS_ATTENTION_LAYERS, CROSS_ATTENTION_LAYER_OPTIONS)
_validate_choices('CROSS_ATTENTION_HEADS', CROSS_ATTENTION_HEADS, CROSS_ATTENTION_HEAD_OPTIONS)

if EPOCHS < 1:
    raise ValueError('EPOCHS must be at least 1.')
if not 0 <= VAL_FRACTION < 1:
    raise ValueError('VAL_FRACTION must be in the range [0, 1).')
if N_FOLDS is not None and FOLD_INDEX is None or N_FOLDS is None and FOLD_INDEX is not None:
    raise ValueError('N_FOLDS and FOLD_INDEX must both be blank or both be set.')
if N_FOLDS is not None and (N_FOLDS < 2 or not 0 <= FOLD_INDEX < N_FOLDS):
    raise ValueError('N_FOLDS must be at least 2 and FOLD_INDEX must be in [0, N_FOLDS - 1].')
if any(value <= 0 for value in LEARNING_RATES):
    raise ValueError('Learning rates must be positive.')
if any(value < 0 for value in WEIGHT_DECAYS):
    raise ValueError('Weight decay values cannot be negative.')
if any(value < 1 for value in BATCH_SIZES):
    raise ValueError('Batch sizes must be at least 1.')
if any(value < 1 for value in GVP_LAYER_VALUES):
    raise ValueError('GVP layer values must be at least 1.')
if any(value < 1 for value in HIDDEN_S_VALUES + HIDDEN_V_VALUES + EDGE_HIDDEN_VALUES + ESM_FUSION_DIM_VALUES + HEAD_MLP_LAYER_VALUES):
    raise ValueError('Hidden dimensions and layer counts must be at least 1.')
if any(value <= 0 for value in EDGE_RADIUS_VALUES):
    raise ValueError('Edge radius values must be positive.')
if NODE_RBF_SIGMA <= 0 or EDGE_RBF_SIGMA <= 0:
    raise ValueError('RBF sigma values must be positive.')
if any(value < 0 for value in EC_CONTRASTIVE_WEIGHTS):
    raise ValueError('EC contrastive weights cannot be negative.')
if EC_CONTRASTIVE_TEMPERATURE <= 0:
    raise ValueError('EC contrastive temperature must be positive.')
if not 0 <= CROSS_ATTENTION_DROPOUT < 1:
    raise ValueError('CROSS_ATTENTION_DROPOUT must be in the range [0, 1).')
if not 0 <= EARLY_ESM_DROPOUT < 1:
    raise ValueError('EARLY_ESM_DROPOUT must be in the range [0, 1).')
if LR_STEP_SIZE < 1:
    raise ValueError('LR_STEP_SIZE must be at least 1.')
if LR_DECAY_GAMMA <= 0:
    raise ValueError('LR_DECAY_GAMMA must be positive.')
if not 0 <= METAL_LABEL_SMOOTHING < 1:
    raise ValueError('METAL_LABEL_SMOOTHING must be in the range [0, 1).')
if any(value < 0 for value in [METAL_LOSS_WEIGHT, EC_LOSS_WEIGHT, MN_LOSS_MULTIPLIER, CU_LOSS_MULTIPLIER, ZN_LOSS_MULTIPLIER, FE_LOSS_MULTIPLIER, CO_LOSS_MULTIPLIER, NI_LOSS_MULTIPLIER, CLASS_VIII_LOSS_MULTIPLIER]):
    raise ValueError('Loss weights and class multipliers cannot be negative.')
if MAX_SWEEP_RUNS < 1:
    raise ValueError('MAX_SWEEP_RUNS must be at least 1.')

if 'radius_only' in RING_EDGES_MODES and len(RING_EDGES_MODES) == 1:
    REQUIRE_RING_EDGES = False
    PREPARE_MISSING_RING_EDGES = False
elif 'generate_missing_ring' not in RING_EDGES_MODES:
    PREPARE_MISSING_RING_EDGES = False

# Backward-compatible first-value aliases used by command-preview helpers.
TASK_MODE = TASK_MODES[0]
MODEL_PRESET = MODEL_PRESETS[0]
RING_EDGES_MODE = RING_EDGES_MODES[0]
LEARNING_RATE = LEARNING_RATES[0]
WEIGHT_DECAY = WEIGHT_DECAYS[0]
BATCH_SIZE = BATCH_SIZES[0]
SEED = SEEDS[0]
NODE_FEATURE_SET = NODE_FEATURE_SETS[0]
GVP_LAYERS = GVP_LAYER_VALUES[0]
HIDDEN_S = HIDDEN_S_VALUES[0]
HIDDEN_V = HIDDEN_V_VALUES[0]
EDGE_HIDDEN = EDGE_HIDDEN_VALUES[0]
EDGE_RADIUS = EDGE_RADIUS_VALUES[0]
ESM_FUSION_DIM = ESM_FUSION_DIM_VALUES[0]
HEAD_MLP_LAYERS = HEAD_MLP_LAYER_VALUES[0]
EC_LABEL_DEPTH = EC_LABEL_DEPTHS[0]
EC_CONTRASTIVE_WEIGHT = EC_CONTRASTIVE_WEIGHTS[0]
LR_SCHEDULE = LR_SCHEDULES[0]

_config_summary = {
    'paths': {
        'train_dir': str(train_dir),
        'train_csv': str(train_csv),
        'test_dir': str(test_dir),
        'test_csv': str(test_csv),
        'runs_dir': str(local_runs_dir),
        'external_features_root_dir': EXTERNAL_FEATURES_ROOT_DIR or '(code default)',
        'external_feature_source': EXTERNAL_FEATURE_SOURCE,
    },
    'single_value_settings': {
        'EPOCHS': EPOCHS,
        'VAL_FRACTION': VAL_FRACTION,
        'DEVICE': DEVICE,
        'RUN_NAME': RUN_NAME or '(auto)',
        'SPLIT_BY': SPLIT_BY,
        'N_FOLDS': N_FOLDS,
        'FOLD_INDEX': FOLD_INDEX,
        'SELECTION_METRIC_OVERRIDE': SELECTION_METRIC_OVERRIDE or '(task default)',
        'DETERMINISTIC': DETERMINISTIC,
        'SAVE_EPOCH_CHECKPOINTS': SAVE_EPOCH_CHECKPOINTS,
        'RUN_HELD_OUT_TEST_EVAL': RUN_HELD_OUT_TEST_EVAL,
        'ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG': ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG,
        'ESM_DIM': ESM_DIM,
        'NODE_RBF_SIGMA': NODE_RBF_SIGMA,
        'EDGE_RBF_SIGMA': EDGE_RBF_SIGMA,
        'NODE_RBF_USE_RAW_DISTANCES': NODE_RBF_USE_RAW_DISTANCES,
        'DISABLE_ESM_BRANCH': DISABLE_ESM_BRANCH,
        'CROSS_ATTENTION_DROPOUT': CROSS_ATTENTION_DROPOUT,
        'CROSS_ATTENTION_NEIGHBORHOOD': CROSS_ATTENTION_NEIGHBORHOOD,
        'CROSS_ATTENTION_BIDIRECTIONAL': CROSS_ATTENTION_BIDIRECTIONAL,
        'USE_ESM_EMBEDDINGS': USE_ESM_EMBEDDINGS,
        'ESM_EMBEDDINGS_DIR': ESM_EMBEDDINGS_DIR,
        'PREPARE_MISSING_ESM_EMBEDDINGS': PREPARE_MISSING_ESM_EMBEDDINGS,
        'ALLOW_MISSING_ESM_EMBEDDINGS': ALLOW_MISSING_ESM_EMBEDDINGS,
        'ALLOW_MISSING_EXTERNAL_FEATURES': ALLOW_MISSING_EXTERNAL_FEATURES,
        'EARLY_ESM_DIM': EARLY_ESM_DIM,
        'EARLY_ESM_DROPOUT': EARLY_ESM_DROPOUT,
        'EARLY_ESM_RAW': EARLY_ESM_RAW,
        'EARLY_ESM_SCOPE': EARLY_ESM_SCOPE,
        'RING_EXE_PATH': RING_EXE_PATH,
        'RING_EDGE_OUTPUT_DIR': RING_EDGE_OUTPUT_DIR,
        'PRECOMPUTED_RING_EDGES_DIR': PRECOMPUTED_RING_EDGES_DIR,
        'REQUIRE_RING_EDGES': REQUIRE_RING_EDGES,
        'PREPARE_MISSING_RING_EDGES': PREPARE_MISSING_RING_EDGES,
        'METAL_LOSS_FUNCTION': METAL_LOSS_FUNCTION,
        'METAL_LOSS_WEIGHT': METAL_LOSS_WEIGHT,
        'EC_LOSS_WEIGHT': EC_LOSS_WEIGHT,
        'EC_GROUP_WEIGHTING': EC_GROUP_WEIGHTING,
    },
    'experiment_grid': {
        'TASK_MODES': TASK_MODES,
        'MODEL_PRESETS': MODEL_PRESETS,
        'RING_EDGES_MODES': RING_EDGES_MODES,
        'LEARNING_RATES': LEARNING_RATES,
        'WEIGHT_DECAYS': WEIGHT_DECAYS,
        'BATCH_SIZES': BATCH_SIZES,
        'SEEDS': SEEDS,
        'NODE_FEATURE_SETS': NODE_FEATURE_SETS,
        'NODE_FEATURES_OMIT': NODE_FEATURES_OMIT,
        'EC_LABEL_DEPTHS': EC_LABEL_DEPTHS,
        'EC_CONTRASTIVE_WEIGHTS': EC_CONTRASTIVE_WEIGHTS,
        'GVP_LAYER_VALUES': GVP_LAYER_VALUES,
        'HIDDEN_S_VALUES': HIDDEN_S_VALUES,
        'HIDDEN_V_VALUES': HIDDEN_V_VALUES,
        'EDGE_HIDDEN_VALUES': EDGE_HIDDEN_VALUES,
        'EDGE_RADIUS_VALUES': EDGE_RADIUS_VALUES,
        'ESM_FUSION_DIM_VALUES': ESM_FUSION_DIM_VALUES,
        'HEAD_MLP_LAYER_VALUES': HEAD_MLP_LAYER_VALUES,
        'CROSS_ATTENTION_LAYERS': CROSS_ATTENTION_LAYERS,
        'CROSS_ATTENTION_HEADS': CROSS_ATTENTION_HEADS,
        'LR_SCHEDULES': LR_SCHEDULES,
        'MAX_SWEEP_RUNS': MAX_SWEEP_RUNS,
    },
    'execution': {
        'training_behavior': 'inferred from planned run count',
        'COPY_OUTPUTS_TO_DRIVE': COPY_OUTPUTS_TO_DRIVE,
    },
}

print('Final configuration parsed from widget values:')
print(json.dumps(_config_summary, indent=2, sort_keys=True))


ValueError: N_FOLDS must be blank or an integer, got 'None'.

## 8. Build Commands and Planned Experiments

Review the first planned grid command and the full planned experiment table before running. The helper functions below build CLI commands only; model training still happens exclusively in `src/train.py`.


In [ ]:
import csv
import hashlib
import itertools
import json
import os
import re
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

local_runs_dir.mkdir(parents=True, exist_ok=True)

VALID_RING_EDGE_MODES = {'radius_only', 'radius_plus_precomputed_ring', 'generate_missing_ring'}
GVP_LAYER_VALUES = list(globals().get('GVP_LAYER_VALUES', [globals().get('GVP_LAYERS', 4)]))
HIDDEN_S_VALUES = list(globals().get('HIDDEN_S_VALUES', [globals().get('HIDDEN_S', 128)]))
HIDDEN_V_VALUES = list(globals().get('HIDDEN_V_VALUES', [globals().get('HIDDEN_V', 16)]))
EDGE_HIDDEN_VALUES = list(globals().get('EDGE_HIDDEN_VALUES', [globals().get('EDGE_HIDDEN', 64)]))
EDGE_RADIUS_VALUES = list(globals().get('EDGE_RADIUS_VALUES', [globals().get('EDGE_RADIUS', 8.0)]))
ESM_FUSION_DIM_VALUES = list(globals().get('ESM_FUSION_DIM_VALUES', [globals().get('ESM_FUSION_DIM', 128)]))
HEAD_MLP_LAYER_VALUES = list(globals().get('HEAD_MLP_LAYER_VALUES', [globals().get('HEAD_MLP_LAYERS', 2)]))
ESM_DIM = int(globals().get('ESM_DIM', 960))
NODE_RBF_SIGMA = float(globals().get('NODE_RBF_SIGMA', 0.75))
EDGE_RBF_SIGMA = float(globals().get('EDGE_RBF_SIGMA', 0.75))
NODE_RBF_USE_RAW_DISTANCES = bool(globals().get('NODE_RBF_USE_RAW_DISTANCES', False))
DISABLE_ESM_BRANCH = bool(globals().get('DISABLE_ESM_BRANCH', False))
EARLY_ESM_DIM = int(globals().get('EARLY_ESM_DIM', 32))
EARLY_ESM_DROPOUT = float(globals().get('EARLY_ESM_DROPOUT', 0.2))
EARLY_ESM_RAW = bool(globals().get('EARLY_ESM_RAW', False))
EARLY_ESM_SCOPE = globals().get('EARLY_ESM_SCOPE', 'all')
EXTERNAL_FEATURES_ROOT_DIR = globals().get('EXTERNAL_FEATURES_ROOT_DIR', '')
EXTERNAL_FEATURE_SOURCE = globals().get('EXTERNAL_FEATURE_SOURCE', 'auto')
SELECTION_METRIC_OVERRIDE = globals().get('SELECTION_METRIC_OVERRIDE', '')
N_FOLDS = globals().get('N_FOLDS', None)
FOLD_INDEX = globals().get('FOLD_INDEX', None)
DETERMINISTIC = bool(globals().get('DETERMINISTIC', False))
SAVE_EPOCH_CHECKPOINTS = bool(globals().get('SAVE_EPOCH_CHECKPOINTS', False))
ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG = bool(globals().get('ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG', False))
BALANCE_METAL_SITE_SYMBOLS = bool(globals().get('BALANCE_METAL_SITE_SYMBOLS', False))
REQUIRE_ALL_TASK_CLASSES = bool(globals().get('REQUIRE_ALL_TASK_CLASSES', False))
METAL_LOSS_FUNCTION = globals().get('METAL_LOSS_FUNCTION', 'cross_entropy')
METAL_FOCAL_GAMMA = float(globals().get('METAL_FOCAL_GAMMA', 2.0))
METAL_LABEL_SMOOTHING = float(globals().get('METAL_LABEL_SMOOTHING', 0.0))
METAL_LOSS_WEIGHT = float(globals().get('METAL_LOSS_WEIGHT', 1.0))
EC_LOSS_WEIGHT = float(globals().get('EC_LOSS_WEIGHT', 1.0))
MN_LOSS_MULTIPLIER = float(globals().get('MN_LOSS_MULTIPLIER', 1.0))
CU_LOSS_MULTIPLIER = float(globals().get('CU_LOSS_MULTIPLIER', 1.0))
ZN_LOSS_MULTIPLIER = float(globals().get('ZN_LOSS_MULTIPLIER', 1.0))
FE_LOSS_MULTIPLIER = float(globals().get('FE_LOSS_MULTIPLIER', 1.0))
CO_LOSS_MULTIPLIER = float(globals().get('CO_LOSS_MULTIPLIER', 1.0))
NI_LOSS_MULTIPLIER = float(globals().get('NI_LOSS_MULTIPLIER', 1.0))
CLASS_VIII_LOSS_MULTIPLIER = float(globals().get('CLASS_VIII_LOSS_MULTIPLIER', 1.0))
EC_GROUP_WEIGHTING = globals().get('EC_GROUP_WEIGHTING', 'structure_id')
EC_CONTRASTIVE_TEMPERATURE = float(globals().get('EC_CONTRASTIVE_TEMPERATURE', 0.1))
UNSUPPORTED_METAL_POLICY = globals().get('UNSUPPORTED_METAL_POLICY', 'error')
INVALID_STRUCTURE_POLICY = globals().get('INVALID_STRUCTURE_POLICY', 'skip')
STOP_ON_FIRST_FAILURE = globals().get('STOP_ON_FIRST_FAILURE', True)
SKIP_EXISTING_RUNS = globals().get('SKIP_EXISTING_RUNS', True)
if 'RING_EDGE_MODE_LABEL_TO_VALUE' not in globals():
    RING_EDGE_MODE_LABEL_TO_VALUE = {
        'Radius only (no RING interaction edges)': 'radius_only',
        'Radius + existing RING edge files': 'radius_plus_precomputed_ring',
        'Radius + generate missing RING edge files': 'generate_missing_ring',
    }
    RING_EDGE_MODE_VALUE_TO_LABEL = {value: label for label, value in RING_EDGE_MODE_LABEL_TO_VALUE.items()}
if 'normalize_ring_edges_mode' not in globals():
    def normalize_ring_edges_mode(value):
        text = str(value).strip()
        if text in RING_EDGE_MODE_LABEL_TO_VALUE:
            return RING_EDGE_MODE_LABEL_TO_VALUE[text]
        if text in RING_EDGE_MODE_VALUE_TO_LABEL:
            return text
        valid = ', '.join(RING_EDGE_MODE_LABEL_TO_VALUE)
        raise ValueError(f'Unsupported RING edge mode {value!r}. Choose one of: {valid}')
if 'ring_edges_mode_label' not in globals():
    def ring_edges_mode_label(value):
        mode = normalize_ring_edges_mode(value)
        return RING_EDGE_MODE_VALUE_TO_LABEL[mode]


def slugify(value):
    text = str(value).lower().replace('+', 'plus')
    text = re.sub(r'[^a-z0-9]+', '_', text).strip('_')
    return text or 'value'


def format_float_token(value):
    token = f'{float(value):.0e}' if abs(float(value)) < 0.001 else f'{float(value):g}'
    return token.replace('+', '').replace('-', 'm').replace('.', 'p')


def config_hash(config):
    payload = json.dumps(config, sort_keys=True, default=str)
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()[:8]


def command_string(cmd):
    return ' '.join(shlex.quote(str(part)) for part in cmd)


def make_safe_run_name(config, requested_name=None):
    if requested_name:
        return slugify(requested_name)
    parts = [
        slugify(config['task_mode']),
        slugify(config['model_preset']),
        f"edges{slugify(config['ring_edges_mode'])}",
        f"lr{format_float_token(config['learning_rate'])}",
        f"wd{format_float_token(config['weight_decay'])}",
        f"bs{config['batch_size']}",
        f"seed{config['seed']}",
        f"node{slugify(config['node_feature_set'])}",
    ]
    if config.get('fusion_mode'):
        parts.insert(2, slugify(config['fusion_mode']))
    if config['task'] == 'ec':
        parts.extend([
            f"ecdepth{config['ec_label_depth']}",
            f"eccont{format_float_token(config['ec_contrastive_weight'])}",
        ])
    if config['model_architecture'] in {'gvp', 'only_gvp', 'simple_gnn_esm'}:
        parts.extend([
            f"layers{config['gvp_layers']}",
            f"hs{config['hidden_s']}",
            f"hv{config['hidden_v']}",
            f"eh{config['edge_hidden']}",
            f"r{format_float_token(config['edge_radius'])}",
        ])
    if config['uses_esm']:
        parts.append(f"esmf{config['esm_fusion_dim']}")
    parts.append(f"head{config['head_mlp_layers']}")
    if config.get('fusion_mode') == 'cross_modal_attention':
        parts.extend([
            f"xal{config['cross_attention_layers']}",
            f"xah{config['cross_attention_heads']}",
        ])
    parts.append(f"h{config_hash(config)}")
    return '_'.join(parts)


def path_from_control(value, *, base=repo_dir):
    path = Path(str(value)).expanduser()
    if path.is_absolute():
        return path
    return (Path(base) / path).resolve()


def resolve_ring_exe_control(value):
    raw_path = Path(str(value)).expanduser()
    if not str(raw_path):
        return raw_path
    if raw_path.is_absolute():
        return raw_path
    ring_path = raw_path
    drive_candidate = drive_data_dir / ring_path
    repo_candidate = repo_dir / drive_data_dir.name / ring_path
    drive_mount_available = str(drive_data_dir).startswith('/content/drive/') and Path('/content/drive').exists()
    candidates = [
        (repo_dir / ring_path).resolve(),
        drive_candidate.resolve(),
        repo_candidate.resolve(),
        (drive_data_dir / ring_path).resolve(),
        (drive_data_dir.parent / ring_path).resolve(),
    ]
    if drive_mount_available:
        candidates.insert(0, drive_candidate.resolve())
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def sample_files(root, pattern, limit=5):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(root.rglob(pattern))[:limit]


def colab_local_path_warning(path):
    text = str(path)
    return RUN_MODE == 'colab' and text.startswith('/home/')


def count_files(root, pattern):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(1 for _ in root.rglob(pattern))


def build_run_config(
    *,
    task_mode,
    model_preset,
    ring_edges_mode,
    learning_rate,
    weight_decay,
    batch_size,
    seed,
    node_feature_set,
    ec_label_depth,
    ec_contrastive_weight,
    cross_attention_layers,
    cross_attention_heads,
    lr_schedule,
    gvp_layers=None,
    hidden_s=None,
    hidden_v=None,
    edge_hidden=None,
    edge_radius=None,
    esm_fusion_dim=None,
    head_mlp_layers=None,
    requested_name=None,
):
    ring_edges_mode = normalize_ring_edges_mode(ring_edges_mode)
    if ring_edges_mode not in VALID_RING_EDGE_MODES:
        raise ValueError(f'Unsupported RING_EDGES_MODE: {ring_edges_mode}')
    task, default_selection_metric = task_map[task_mode]
    selection_metric = SELECTION_METRIC_OVERRIDE or default_selection_metric
    preset = dict(preset_map[model_preset])
    uses_esm = bool(preset.get('uses_esm'))
    gvp_layers = int(gvp_layers if gvp_layers is not None else globals().get('GVP_LAYERS', 4))
    hidden_s = int(hidden_s if hidden_s is not None else globals().get('HIDDEN_S', 128))
    hidden_v = int(hidden_v if hidden_v is not None else globals().get('HIDDEN_V', 16))
    edge_hidden = int(edge_hidden if edge_hidden is not None else globals().get('EDGE_HIDDEN', 64))
    edge_radius = float(edge_radius if edge_radius is not None else globals().get('EDGE_RADIUS', 8.0))
    esm_fusion_dim = int(esm_fusion_dim if esm_fusion_dim is not None else globals().get('ESM_FUSION_DIM', 128))
    head_mlp_layers = int(head_mlp_layers if head_mlp_layers is not None else globals().get('HEAD_MLP_LAYERS', 2))
    esm_embeddings_dir = str(path_from_control(ESM_EMBEDDINGS_DIR)) if uses_esm else ''
    ring_output_dir = str(path_from_control(RING_EDGE_OUTPUT_DIR))
    precomputed_ring_dir = str(path_from_control(PRECOMPUTED_RING_EDGES_DIR))
    disabled_ring_dir = str(local_runs_dir / '_radius_only_no_ring_edges')
    if ring_edges_mode == 'radius_only':
        ring_features_dir = disabled_ring_dir
    elif ring_edges_mode == 'radius_plus_precomputed_ring':
        ring_features_dir = precomputed_ring_dir
    else:
        ring_features_dir = ring_output_dir
    prepare_ring = bool(PREPARE_MISSING_RING_EDGES and ring_edges_mode == 'generate_missing_ring')
    require_ring = bool(REQUIRE_RING_EDGES and ring_edges_mode != 'radius_only')
    config = {
        'task_mode': task_mode,
        'task': task,
        'selection_metric': selection_metric,
        'model_preset': model_preset,
        'model_architecture': preset['model_architecture'],
        'fusion_mode': preset.get('fusion_mode'),
        'early_esm_dim': EARLY_ESM_DIM,
        'early_esm_dropout': EARLY_ESM_DROPOUT,
        'epochs': int(EPOCHS),
        'batch_size': int(batch_size),
        'learning_rate': float(learning_rate),
        'weight_decay': float(weight_decay),
        'seed': int(seed),
        'val_fraction': float(VAL_FRACTION),
        'device': DEVICE,
        'split_by': SPLIT_BY,
        'node_feature_set': node_feature_set,
        'omit_node_features': list(globals().get('NODE_FEATURES_OMIT', [])),
        'gvp_layers': gvp_layers,
        'hidden_s': hidden_s,
        'hidden_v': hidden_v,
        'edge_hidden': edge_hidden,
        'edge_radius': edge_radius,
        'esm_fusion_dim': esm_fusion_dim,
        'head_mlp_layers': head_mlp_layers,
        'esm_dim': ESM_DIM,
        'node_rbf_sigma': NODE_RBF_SIGMA,
        'edge_rbf_sigma': EDGE_RBF_SIGMA,
        'node_rbf_use_raw_distances': NODE_RBF_USE_RAW_DISTANCES,
        'disable_esm_branch': bool(DISABLE_ESM_BRANCH),
        'ec_label_depth': int(ec_label_depth),
        'ec_contrastive_weight': float(ec_contrastive_weight),
        'ec_contrastive_temperature': EC_CONTRASTIVE_TEMPERATURE,
        'ec_group_weighting': EC_GROUP_WEIGHTING,
        'lr_schedule': lr_schedule,
        'lr_step_size': int(LR_STEP_SIZE),
        'lr_decay_gamma': float(LR_DECAY_GAMMA),
        'cross_attention_layers': int(cross_attention_layers),
        'cross_attention_heads': int(cross_attention_heads),
        'cross_attention_dropout': float(CROSS_ATTENTION_DROPOUT),
        'cross_attention_neighborhood': CROSS_ATTENTION_NEIGHBORHOOD,
        'cross_attention_bidirectional': bool(CROSS_ATTENTION_BIDIRECTIONAL),
        'early_esm_raw': EARLY_ESM_RAW,
        'early_esm_scope': EARLY_ESM_SCOPE,
        'n_folds': N_FOLDS,
        'fold_index': FOLD_INDEX,
        'deterministic': DETERMINISTIC,
        'save_epoch_checkpoints': SAVE_EPOCH_CHECKPOINTS,
        'allow_train_loss_test_eval_debug': ALLOW_TRAIN_LOSS_TEST_EVAL_DEBUG,
        'balance_metal_site_symbols': BALANCE_METAL_SITE_SYMBOLS,
        'require_all_task_classes': REQUIRE_ALL_TASK_CLASSES,
        'metal_loss_function': METAL_LOSS_FUNCTION,
        'metal_focal_gamma': METAL_FOCAL_GAMMA,
        'metal_label_smoothing': METAL_LABEL_SMOOTHING,
        'metal_loss_weight': METAL_LOSS_WEIGHT,
        'ec_loss_weight': EC_LOSS_WEIGHT,
        'mn_loss_multiplier': MN_LOSS_MULTIPLIER,
        'cu_loss_multiplier': CU_LOSS_MULTIPLIER,
        'zn_loss_multiplier': ZN_LOSS_MULTIPLIER,
        'fe_loss_multiplier': FE_LOSS_MULTIPLIER,
        'co_loss_multiplier': CO_LOSS_MULTIPLIER,
        'ni_loss_multiplier': NI_LOSS_MULTIPLIER,
        'class_viii_loss_multiplier': CLASS_VIII_LOSS_MULTIPLIER,
        'unsupported_metal_policy': UNSUPPORTED_METAL_POLICY,
        'invalid_structure_policy': INVALID_STRUCTURE_POLICY,
        'run_test_eval': bool(RUN_HELD_OUT_TEST_EVAL),
        'uses_esm': uses_esm,
        'use_esm_embeddings': bool(USE_ESM_EMBEDDINGS and uses_esm),
        'esm_embeddings_dir': esm_embeddings_dir,
        'prepare_missing_esm_embeddings': bool(PREPARE_MISSING_ESM_EMBEDDINGS and uses_esm),
        'allow_missing_esm_embeddings': bool(ALLOW_MISSING_ESM_EMBEDDINGS or not uses_esm),
        'allow_missing_external_features': bool(ALLOW_MISSING_EXTERNAL_FEATURES),
        'external_features_root_dir': str(path_from_control(EXTERNAL_FEATURES_ROOT_DIR)) if str(EXTERNAL_FEATURES_ROOT_DIR).strip() else '',
        'external_feature_source': EXTERNAL_FEATURE_SOURCE,
        'ring_edges_mode': ring_edges_mode,
        'use_ring_edges': bool(ring_edges_mode != 'radius_only'),
        'require_ring_edges': require_ring,
        'prepare_missing_ring_edges': prepare_ring,
        'ring_exe_path': str(resolve_ring_exe_control(RING_EXE_PATH)) if ring_edges_mode == 'generate_missing_ring' else '',
        'ring_edge_output_dir': ring_output_dir,
        'precomputed_ring_edges_dir': precomputed_ring_dir if ring_edges_mode == 'radius_plus_precomputed_ring' else '',
        'ring_features_dir': ring_features_dir,
    }
    config['run_name'] = make_safe_run_name(config, requested_name=requested_name)
    config['run_dir'] = local_runs_dir / config['run_name']
    return config


def esm_readiness_issues(config):
    if not config['uses_esm']:
        return []
    issues = []
    embeddings_dir = Path(config['esm_embeddings_dir'])
    embedding_count = count_files(embeddings_dir, '*_esmc.pt')
    if not config['use_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        issues.append('Selected ESM preset has USE_ESM_EMBEDDINGS=False. Set ALLOW_MISSING_ESM_EMBEDDINGS=True to train explicitly without embeddings.')
    if not config['prepare_missing_esm_embeddings'] and not config['allow_missing_esm_embeddings']:
        if not embeddings_dir.exists():
            issues.append(f'ESM embeddings directory does not exist: {embeddings_dir}')
        elif embedding_count == 0:
            issues.append(f'No *_esmc.pt files found under ESM embeddings directory: {embeddings_dir}')
    return issues


def ring_readiness_issues(config):
    issues = []
    mode = config['ring_edges_mode']
    if mode == 'radius_only':
        return issues
    ring_dir = Path(config['ring_features_dir'])
    ring_count = count_files(ring_dir, '*ringEdges')
    if mode == 'radius_plus_precomputed_ring' and config['require_ring_edges']:
        if not ring_dir.exists():
            issues.append(f'Precomputed RING edge directory does not exist: {ring_dir}')
        elif ring_count == 0:
            issues.append(f'No *ringEdges files found under precomputed RING edge directory: {ring_dir}')
    if mode == 'generate_missing_ring' and config['prepare_missing_ring_edges']:
        ring_exe = Path(config['ring_exe_path'])
        if not ring_exe.is_file():
            issues.append(f'RING executable does not exist: {ring_exe}')
        elif not os.access(ring_exe, os.X_OK):
            issues.append(f'RING executable is not executable: {ring_exe}. Run: chmod +x {ring_exe}')
    return issues


def path_readiness_issues(config):
    issues = []
    watched_paths = [train_dir, train_csv, test_dir, test_csv, local_runs_dir]
    if config.get('esm_embeddings_dir'):
        watched_paths.append(config['esm_embeddings_dir'])
    if config.get('ring_features_dir'):
        watched_paths.append(config['ring_features_dir'])
    if config.get('external_features_root_dir'):
        watched_paths.append(config['external_features_root_dir'])
    for value in watched_paths:
        if colab_local_path_warning(value):
            issues.append(f'Colab cannot see local /home paths: {value}')
    if config.get('external_features_root_dir') and not Path(config['external_features_root_dir']).exists() and not config.get('allow_missing_external_features'):
        issues.append(f'External feature root does not exist and missing external features are not allowed: {config["external_features_root_dir"]}')
    if config.get('early_esm_raw'):
        issues.append('Raw early ESM is enabled; this is a high-dimensional advanced ablation and may use much more memory.')
    if config.get('disable_esm_branch') and config.get('uses_esm'):
        issues.append('Disable ESM branch is enabled for an ESM-using preset. Prefer Only-GVP for the standard graph-only baseline.')
    if config.get('run_test_eval') and config.get('selection_metric') == 'train_loss' and not config.get('allow_train_loss_test_eval_debug'):
        issues.append('Held-out test eval with train_loss selection is blocked by src/train.py unless debug override is enabled.')
    return issues


def readiness_issues(config):
    return esm_readiness_issues(config) + ring_readiness_issues(config) + path_readiness_issues(config)


def validate_run_before_training(config):
    issues = readiness_issues(config)
    if issues:
        raise RuntimeError('Pre-training configuration check failed:\n- ' + '\n- '.join(issues))


def print_run_checks(config, *, label):
    embeddings_dir = Path(config['esm_embeddings_dir']) if config['esm_embeddings_dir'] else None
    embedding_files = sample_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else []
    ring_features_dir = Path(config['ring_features_dir'])
    ring_files = sample_files(ring_features_dir, '*ringEdges')
    ring_exe_user_path = path_from_control(RING_EXE_PATH)
    ring_exe = Path(config['ring_exe_path']) if config['ring_exe_path'] else resolve_ring_exe_control(RING_EXE_PATH)
    issues = readiness_issues(config)
    print(f'{label} checks')
    print('  selected model preset:', config['model_preset'])
    print('  selected model uses ESM:', config['uses_esm'])
    print('  GVP layers:', config['gvp_layers'])
    print('  hidden_s:', config['hidden_s'])
    print('  hidden_v:', config['hidden_v'])
    print('  edge_hidden:', config['edge_hidden'])
    print('  edge_radius:', config['edge_radius'])
    print('  esm_fusion_dim:', config['esm_fusion_dim'])
    print('  head_mlp_layers:', config['head_mlp_layers'])
    print('  node_rbf_sigma:', config['node_rbf_sigma'])
    print('  edge_rbf_sigma:', config['edge_rbf_sigma'])
    print('  node_rbf_use_raw_distances:', config['node_rbf_use_raw_distances'])
    print('  ESM embeddings directory:', config['esm_embeddings_dir'] or 'not used by this preset')
    print('  ESM embeddings directory exists:', embeddings_dir.exists() if embeddings_dir is not None else False)
    print('  existing ESM embedding files:', count_files(embeddings_dir, '*_esmc.pt') if embeddings_dir is not None else 0)
    print('  sample ESM embedding files:', [str(path) for path in embedding_files[:3]])
    print('  missing ESM embeddings will be generated:', config['prepare_missing_esm_embeddings'])
    print('  missing ESM embeddings are allowed:', config['allow_missing_esm_embeddings'])
    print('  ESM mode:', 'not used' if not config['uses_esm'] else ('prepare missing' if config['prepare_missing_esm_embeddings'] else 'precomputed/required'))
    print('  --esm-embeddings-dir is being passed:', config['uses_esm'])
    print('  --allow-missing-esm-embeddings is being passed:', config['allow_missing_esm_embeddings'])
    print('  --no-prepare-missing-esm-embeddings is being passed:', (not config['prepare_missing_esm_embeddings']) or not config['uses_esm'])
    print('  RING edge mode:', ring_edges_mode_label(config['ring_edges_mode']), f"[{config['ring_edges_mode']}]")
    print('  RING_EXE_PATH user setting:', str(ring_exe_user_path))
    print('  RING_EXE_PATH resolved:', str(ring_exe))
    print('  RING_EXE_PATH exists:', ring_exe.is_file())
    print('  RING_EXE_PATH executable:', os.access(ring_exe, os.X_OK) if ring_exe.exists() else False)
    print('  RING edge files expected under:', ring_features_dir)
    print('  --use-ring-edges is being passed:', config['use_ring_edges'])
    print('  generated RING edge files will be written under:', config['ring_edge_output_dir'])
    print('  existing RING edge files:', count_files(ring_features_dir, '*ringEdges'))
    print('  sample RING edge files:', [str(path) for path in ring_files[:3]])
    print('  precomputed RING edge files are being used:', config['ring_edges_mode'] == 'radius_plus_precomputed_ring')
    print('  missing RING edges will be generated:', config['prepare_missing_ring_edges'])
    print('  --require-ring-edges is being passed:', config['require_ring_edges'])
    print('  --prepare-missing-ring-edges is being passed:', config['prepare_missing_ring_edges'])
    if issues:
        print('  pre-training warnings/errors:')
        for issue in issues:
            print('   -', issue)


def build_train_command(config):
    cmd = [
        sys.executable, str(repo_dir / 'src' / 'train.py'),
        '--task', config['task'],
        '--structure-dir', str(train_dir),
        '--summary-csv', str(train_csv),
        '--runs-dir', str(local_runs_dir),
        '--run-name', config['run_name'],
        '--model-architecture', config['model_architecture'],
        '--epochs', str(config['epochs']),
        '--batch-size', str(config['batch_size']),
        '--esm-dim', str(config['esm_dim']),
        '--learning-rate', str(config['learning_rate']),
        '--weight-decay', str(config['weight_decay']),
        '--seed', str(config['seed']),
        '--val-fraction', str(config['val_fraction']),
        '--device', config['device'],
        '--split-by', config['split_by'],
        '--selection-metric', config['selection_metric'],
        '--node-feature-set', config['node_feature_set'],
        '--hidden-s', str(config['hidden_s']),
        '--edge-radius', str(config['edge_radius']),
        '--esm-fusion-dim', str(config['esm_fusion_dim']),
        '--head-mlp-layers', str(config['head_mlp_layers']),
        '--node-rbf-sigma', str(config['node_rbf_sigma']),
        '--edge-rbf-sigma', str(config['edge_rbf_sigma']),
        '--metal-loss-function', config['metal_loss_function'],
        '--metal-focal-gamma', str(config['metal_focal_gamma']),
        '--metal-label-smoothing', str(config['metal_label_smoothing']),
        '--metal-loss-weight', str(config['metal_loss_weight']),
        '--ec-loss-weight', str(config['ec_loss_weight']),
        '--mn-loss-multiplier', str(config['mn_loss_multiplier']),
        '--cu-loss-multiplier', str(config['cu_loss_multiplier']),
        '--zn-loss-multiplier', str(config['zn_loss_multiplier']),
        '--fe-loss-multiplier', str(config['fe_loss_multiplier']),
        '--co-loss-multiplier', str(config['co_loss_multiplier']),
        '--ni-loss-multiplier', str(config['ni_loss_multiplier']),
        '--class-viii-loss-multiplier', str(config['class_viii_loss_multiplier']),
        '--unsupported-metal-policy', config['unsupported_metal_policy'],
        '--invalid-structure-policy', config['invalid_structure_policy'],
        '--external-feature-source', config['external_feature_source'],
        '--lr-schedule', config['lr_schedule'],
    ]
    if config.get('omit_node_features'):
        cmd.extend(['--omit-node-features', ','.join(config['omit_node_features'])])
    if config['model_architecture'] in {'gvp', 'only_gvp', 'simple_gnn_esm'}:
        cmd.extend(['--gvp-layers', str(config['gvp_layers']), '--edge-hidden', str(config['edge_hidden'])])
    if config['model_architecture'] in {'gvp', 'only_gvp'}:
        cmd.extend(['--hidden-v', str(config['hidden_v'])])
    if config['node_rbf_use_raw_distances']:
        cmd.append('--node-rbf-use-raw-distances')
    if config['disable_esm_branch']:
        cmd.append('--disable-esm-branch')
    if config['deterministic']:
        cmd.append('--deterministic')
    if config['save_epoch_checkpoints']:
        cmd.append('--save-epoch-checkpoints')
    if config['allow_train_loss_test_eval_debug']:
        cmd.append('--allow-train-loss-test-eval-debug')
    if config['balance_metal_site_symbols']:
        cmd.append('--balance-metal-site-symbols')
    if config['require_all_task_classes']:
        cmd.append('--require-all-task-classes')
    if config['n_folds'] is not None:
        cmd.extend(['--n-folds', str(config['n_folds']), '--fold-index', str(config['fold_index'])])
    if config['external_features_root_dir']:
        cmd.extend(['--external-features-root-dir', config['external_features_root_dir']])
    if config['lr_schedule'] == 'step':
        cmd.extend(['--lr-step-size', str(config['lr_step_size']), '--lr-decay-gamma', str(config['lr_decay_gamma'])])
    if config.get('fusion_mode'):
        cmd.extend(['--fusion-mode', config['fusion_mode']])
    if config.get('fusion_mode') in {'early_fusion', 'hybrid'}:
        cmd.extend(['--use-early-esm', '--early-esm-dim', str(config['early_esm_dim']), '--early-esm-dropout', str(config['early_esm_dropout']), '--early-esm-scope', config['early_esm_scope']])
        if config['early_esm_raw']:
            cmd.append('--early-esm-raw')
    if config.get('fusion_mode') == 'cross_modal_attention':
        cmd.extend([
            '--cross-attention-layers', str(config['cross_attention_layers']),
            '--cross-attention-heads', str(config['cross_attention_heads']),
            '--cross-attention-dropout', str(config['cross_attention_dropout']),
            '--cross-attention-neighborhood', config['cross_attention_neighborhood'],
        ])
        if config['cross_attention_bidirectional']:
            cmd.append('--cross-attention-bidirectional')
    if config['task'] in {'ec', 'joint'}:
        cmd.extend(['--ec-label-depth', str(config['ec_label_depth'])])
        cmd.extend(['--ec-group-weighting', config['ec_group_weighting']])
        cmd.extend(['--ec-contrastive-weight', str(config['ec_contrastive_weight'])])
        cmd.extend(['--ec-contrastive-temperature', str(config['ec_contrastive_temperature'])])
    if config['run_test_eval']:
        cmd.extend(['--test-structure-dir', str(test_dir), '--test-summary-csv', str(test_csv), '--run-test-eval'])
    if config['uses_esm']:
        cmd.extend(['--esm-embeddings-dir', config['esm_embeddings_dir']])
        if config['allow_missing_esm_embeddings']:
            cmd.append('--allow-missing-esm-embeddings')
        if not config['prepare_missing_esm_embeddings']:
            cmd.append('--no-prepare-missing-esm-embeddings')
    else:
        cmd.extend(['--allow-missing-esm-embeddings', '--no-prepare-missing-esm-embeddings'])
    if config['allow_missing_external_features']:
        cmd.append('--allow-missing-external-features')
    if config['use_ring_edges']:
        cmd.append('--use-ring-edges')
    if config['require_ring_edges']:
        cmd.append('--require-ring-edges')
    if config['prepare_missing_ring_edges']:
        cmd.append('--prepare-missing-ring-edges')
    else:
        cmd.append('--no-prepare-missing-ring-edges')
    if config['use_ring_edges']:
        cmd.extend(['--ring-features-dir', config['ring_features_dir']])
    return cmd


def stream_command(cmd, *, cwd, env):
    process = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
    return process.wait()


SWEEP_STATUS_COLUMNS = [
    'run_index', 'total_runs', 'run_name', 'task', 'task_mode', 'model_preset',
    'model_architecture', 'fusion_mode', 'learning_rate', 'weight_decay',
    'batch_size', 'seed', 'node_feature_set', 'gvp_layers', 'hidden_s',
    'hidden_v', 'edge_hidden', 'edge_radius', 'esm_fusion_dim', 'head_mlp_layers',
    'ec_label_depth',
    'ec_contrastive_weight', 'ec_group_weighting', 'metal_loss_function', 'metal_loss_weight', 'ec_loss_weight', 'cross_attention_layers', 'cross_attention_heads',
    'lr_schedule', 'selection_metric', 'val_fraction', 'split_by', 'uses_esm',
    'esm_embeddings_dir', 'prepare_missing_esm_embeddings', 'allow_missing_esm_embeddings',
    'ring_edges_mode', 'use_ring_edges', 'require_ring_edges', 'prepare_missing_ring_edges',
    'ring_exe_path', 'ring_edge_output_dir', 'ring_features_dir',
    'status', 'start_time', 'end_time', 'update_time', 'run_dir',
    'selected_checkpoint', 'selected_metric_value', 'test_summary', 'command', 'error_message',
]


def write_sweep_status(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=SWEEP_STATUS_COLUMNS)
        writer.writeheader()
        for row in rows:
            writer.writerow({column: row.get(column, '') for column in SWEEP_STATUS_COLUMNS})


def summarize_completed_run(run_dir):
    def read_json(path):
        try:
            return json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
        except Exception as exc:
            return {'_read_error': str(exc)}
    run_config = read_json(run_dir / 'run_config.json')
    run_metadata = read_json(run_dir / 'run_metadata.json')
    test_report = read_json(run_dir / 'test_report.json')
    selected_checkpoint = run_metadata.get('selected_checkpoint') or run_config.get('selected_checkpoint')
    selection_metric = run_metadata.get('selection_metric') or run_config.get('selection_metric')
    selected_metric_value = run_metadata.get('selected_metric_value') or run_config.get('selected_metric_value')
    metrics = test_report.get('metrics', {}) if isinstance(test_report.get('metrics'), dict) else {}
    priority = [
        'test_metal_balanced_acc', 'test_metal_collapsed4_balanced_acc',
        'test_ec_group_balanced_acc', 'test_ec_group_level_1_balanced_acc',
        'test_ec_level_1_balanced_acc', 'test_loss',
    ]
    test_summary = {key: metrics[key] for key in priority if key in metrics}
    return {
        'selected_checkpoint': selected_checkpoint,
        'selection_metric': selection_metric,
        'selected_metric_value': selected_metric_value,
        'test_summary': test_summary,
    }


def run_report(runs_dir, out_csv, out_figure):
    report_cmd = [
        sys.executable, str(repo_dir / 'src' / 'report_runs.py'),
        '--runs-dir', str(runs_dir),
        '--out-csv', str(out_csv),
        '--out-figure', str(out_figure),
    ]
    print('Report command:')
    print(command_string(report_cmd))
    return stream_command(report_cmd, cwd=repo_dir, env=base_env)


def build_sweep_runs():
    planned = []
    non_ec_tasks = [task for task in TASK_MODES if task not in {'ec_prediction', 'joint_metal_ec'}]
    if non_ec_tasks and (len(EC_LABEL_DEPTHS) > 1 or len(EC_CONTRASTIVE_WEIGHTS) > 1):
        print(
            f'Note: EC_LABEL_DEPTHS={EC_LABEL_DEPTHS} and EC_CONTRASTIVE_WEIGHTS={EC_CONTRASTIVE_WEIGHTS} '
            f'are ignored for non-EC task modes {non_ec_tasks}. '
            'They only expand the sweep for ec_prediction and joint_metal_ec tasks.'
        )
    base_product = itertools.product(
        TASK_MODES,
        MODEL_PRESETS,
        RING_EDGES_MODES,
        LEARNING_RATES,
        WEIGHT_DECAYS,
        BATCH_SIZES,
        SEEDS,
        NODE_FEATURE_SETS,
        LR_SCHEDULES,
        GVP_LAYER_VALUES,
        HIDDEN_S_VALUES,
        HIDDEN_V_VALUES,
        EDGE_HIDDEN_VALUES,
        EDGE_RADIUS_VALUES,
        ESM_FUSION_DIM_VALUES,
        HEAD_MLP_LAYER_VALUES,
    )
    seen_names = set()
    for (
        task_mode, model_preset, ring_edges_mode, lr, wd, batch_size, seed, node_features, lr_schedule,
        gvp_layers, hidden_s, hidden_v, edge_hidden, edge_radius, esm_fusion_dim, head_mlp_layers,
    ) in base_product:
        ec_depth_values = EC_LABEL_DEPTHS if task_mode in {'ec_prediction', 'joint_metal_ec'} else [EC_LABEL_DEPTH]
        ec_weight_values = EC_CONTRASTIVE_WEIGHTS if task_mode in {'ec_prediction', 'joint_metal_ec'} else [EC_CONTRASTIVE_WEIGHT]
        cross_layer_values = CROSS_ATTENTION_LAYERS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_LAYERS[0]]
        cross_head_values = CROSS_ATTENTION_HEADS if model_preset == 'GVP + cross-modal attention' else [CROSS_ATTENTION_HEADS[0]]
        for ec_depth, ec_weight, cross_layers, cross_heads in itertools.product(
            ec_depth_values,
            ec_weight_values,
            cross_layer_values,
            cross_head_values,
        ):
            config = build_run_config(
                task_mode=task_mode,
                model_preset=model_preset,
                ring_edges_mode=ring_edges_mode,
                learning_rate=lr,
                weight_decay=wd,
                batch_size=batch_size,
                seed=seed,
                node_feature_set=node_features,
                ec_label_depth=ec_depth,
                ec_contrastive_weight=ec_weight,
                cross_attention_layers=cross_layers,
                cross_attention_heads=cross_heads,
                lr_schedule=lr_schedule,
                gvp_layers=gvp_layers,
                hidden_s=hidden_s,
                hidden_v=hidden_v,
                edge_hidden=edge_hidden,
                edge_radius=edge_radius,
                esm_fusion_dim=esm_fusion_dim,
                head_mlp_layers=head_mlp_layers,
            )
            if config['run_name'] in seen_names:
                continue
            seen_names.add(config['run_name'])
            config['command'] = build_train_command(config)
            planned.append(config)
    return planned


def planned_table_rows(planned_runs):
    rows = []
    for index, run in enumerate(planned_runs, start=1):
        rows.append({
            'run_index': index,
            'run_name': run['run_name'],
            'task': run['task'],
            'task_mode': run['task_mode'],
            'model_preset': run['model_preset'],
            'model_architecture': run['model_architecture'],
            'fusion_mode': run.get('fusion_mode') or 'none',
            'ring_edges_mode': run['ring_edges_mode'],
            'uses_esm': run['uses_esm'],
            'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
            'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
            'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
            'use_ring_edges': run['use_ring_edges'],
            'require_ring_edges': run['require_ring_edges'],
            'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
            'ring_edge_output_dir': run['ring_edge_output_dir'],
            'learning_rate': run['learning_rate'],
            'weight_decay': run['weight_decay'],
            'batch_size': run['batch_size'],
            'seed': run['seed'],
            'node_feature_set': run['node_feature_set'],
            'gvp_layers': run['gvp_layers'],
            'hidden_s': run['hidden_s'],
            'hidden_v': run['hidden_v'],
            'edge_hidden': run['edge_hidden'],
            'edge_radius': run['edge_radius'],
            'esm_fusion_dim': run['esm_fusion_dim'],
            'head_mlp_layers': run['head_mlp_layers'],
            'ec_label_depth': run['ec_label_depth'] if run['task'] in {'ec', 'joint'} else 'NA',
            'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] in {'ec', 'joint'} else 'NA',
            'ec_group_weighting': run['ec_group_weighting'] if run['task'] in {'ec', 'joint'} else 'NA',
            'metal_loss_function': run['metal_loss_function'],
            'metal_loss_weight': run['metal_loss_weight'],
            'ec_loss_weight': run['ec_loss_weight'],
            'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
            'selection_metric': run['selection_metric'],
            'run_dir': str(run['run_dir']),
        })
    return rows


def status_row_for_run(index, total_runs, run):
    return {
        'run_index': index,
        'total_runs': total_runs,
        'run_name': run['run_name'],
        'task': run['task'],
        'task_mode': run['task_mode'],
        'model_preset': run['model_preset'],
        'model_architecture': run['model_architecture'],
        'fusion_mode': run.get('fusion_mode') or 'none',
        'learning_rate': run['learning_rate'],
        'weight_decay': run['weight_decay'],
        'batch_size': run['batch_size'],
        'seed': run['seed'],
        'node_feature_set': run['node_feature_set'],
        'gvp_layers': run['gvp_layers'],
        'hidden_s': run['hidden_s'],
        'hidden_v': run['hidden_v'],
        'edge_hidden': run['edge_hidden'],
        'edge_radius': run['edge_radius'],
        'esm_fusion_dim': run['esm_fusion_dim'],
        'head_mlp_layers': run['head_mlp_layers'],
        'ec_label_depth': run['ec_label_depth'] if run['task'] in {'ec', 'joint'} else 'NA',
        'ec_contrastive_weight': run['ec_contrastive_weight'] if run['task'] in {'ec', 'joint'} else 'NA',
        'ec_group_weighting': run['ec_group_weighting'] if run['task'] in {'ec', 'joint'} else 'NA',
        'metal_loss_function': run['metal_loss_function'],
        'metal_loss_weight': run['metal_loss_weight'],
        'ec_loss_weight': run['ec_loss_weight'],
        'cross_attention_layers': run['cross_attention_layers'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'cross_attention_heads': run['cross_attention_heads'] if run.get('fusion_mode') == 'cross_modal_attention' else 'NA',
        'lr_schedule': run['lr_schedule'],
        'selection_metric': run['selection_metric'],
        'val_fraction': run['val_fraction'],
        'split_by': run['split_by'],
        'uses_esm': run['uses_esm'],
        'esm_embeddings_dir': run['esm_embeddings_dir'] or 'NA',
        'prepare_missing_esm_embeddings': run['prepare_missing_esm_embeddings'],
        'allow_missing_esm_embeddings': run['allow_missing_esm_embeddings'],
        'ring_edges_mode': run['ring_edges_mode'],
        'use_ring_edges': run['use_ring_edges'],
        'require_ring_edges': run['require_ring_edges'],
        'prepare_missing_ring_edges': run['prepare_missing_ring_edges'],
        'ring_exe_path': run['ring_exe_path'] if run['prepare_missing_ring_edges'] else 'NA',
        'ring_edge_output_dir': run['ring_edge_output_dir'],
        'ring_features_dir': run['ring_features_dir'],
        'status': 'planned',
        'start_time': '',
        'end_time': '',
        'update_time': datetime.now().isoformat(timespec='seconds'),
        'run_dir': str(run['run_dir']),
        'selected_checkpoint': '',
        'selected_metric_value': '',
        'test_summary': '',
        'command': command_string(run['command']),
        'error_message': '',
    }


def mark_status(row, status):
    row['status'] = status
    row['update_time'] = datetime.now().isoformat(timespec='seconds')


planned_sweep_runs = build_sweep_runs()
if not planned_sweep_runs:
    raise ValueError('The experiment grid produced no planned runs. Select at least one value in every grid field.')
if len(planned_sweep_runs) > MAX_SWEEP_RUNS:
    raise ValueError(f'Experiment grid would run {len(planned_sweep_runs)} experiments, above MAX_SWEEP_RUNS={MAX_SWEEP_RUNS}.')

if RUN_NAME and len(planned_sweep_runs) == 1:
    single_named_run = dict(planned_sweep_runs[0])
    single_named_run['run_name'] = slugify(RUN_NAME)
    single_named_run['run_dir'] = local_runs_dir / single_named_run['run_name']
    single_named_run['command'] = build_train_command(single_named_run)
    planned_sweep_runs[0] = single_named_run
elif RUN_NAME and len(planned_sweep_runs) > 1:
    print('Run name is ignored because the experiment grid has more than one planned run; grid run names stay auto-generated.')

single_run_config = planned_sweep_runs[0]
train_cmd = single_run_config['command']
target_run_dir = single_run_config['run_dir']
RUN_NAME = single_run_config['run_name']

def infer_execution_mode(planned_count):
    return 'sweep' if planned_count > 1 else 'single_run'

EFFECTIVE_EXECUTION_MODE = infer_execution_mode(len(planned_sweep_runs))
RUN_SINGLE_MODE = EFFECTIVE_EXECUTION_MODE == 'single_run'
RUN_SWEEP_MODE = EFFECTIVE_EXECUTION_MODE == 'sweep'

planned_sweep_table = planned_table_rows(planned_sweep_runs)
sweep_status_path = local_runs_dir / 'sweep_status.csv'
final_comparison_csv = local_runs_dir / 'sweep_comparison.csv'
final_comparison_figure = local_runs_dir / 'sweep_comparison.png'
completed_run_dirs = []
failed_run_dirs = []

os.environ['RING_EXE_PATH'] = str(resolve_ring_exe_control(RING_EXE_PATH))
os.environ['RING_FEATURES_DIR'] = single_run_config['ring_features_dir']
base_env = os.environ.copy()
base_env['PYTHONPATH'] = str(repo_dir / 'src') + os.pathsep + base_env.get('PYTHONPATH', '')


def env_for_run(config):
    run_env = base_env.copy()
    run_env['RING_EXE_PATH'] = config['ring_exe_path'] or str(resolve_ring_exe_control(RING_EXE_PATH))
    run_env['RING_FEATURES_DIR'] = config['ring_features_dir']
    return run_env


resolved_configuration_summary = {
    'selected_task': single_run_config['task'],
    'task_mode': single_run_config['task_mode'],
    'selected_model_architecture': single_run_config['model_architecture'],
    'model_preset': single_run_config['model_preset'],
    'fusion_mode': single_run_config.get('fusion_mode') or 'none',
    'esm_mode': 'not used' if not single_run_config['uses_esm'] else ('prepare missing' if single_run_config['prepare_missing_esm_embeddings'] else 'precomputed/required'),
    'esm_embeddings_dir': single_run_config['esm_embeddings_dir'] or 'not used',
    'ring_mode': single_run_config['ring_edges_mode'],
    'ring_features_dir': single_run_config['ring_features_dir'],
    'require_ring_edges': single_run_config['require_ring_edges'],
    'prepare_missing_ring_edges': single_run_config['prepare_missing_ring_edges'],
    'prepare_missing_esm_embeddings': single_run_config['prepare_missing_esm_embeddings'],
    'allow_missing_esm_embeddings': single_run_config['allow_missing_esm_embeddings'],
    'training_hyperparameters': {
        'epochs': single_run_config['epochs'],
        'batch_size': single_run_config['batch_size'],
        'learning_rate': single_run_config['learning_rate'],
        'weight_decay': single_run_config['weight_decay'],
        'seed': single_run_config['seed'],
        'val_fraction': single_run_config['val_fraction'],
        'lr_schedule': single_run_config['lr_schedule'],
    },
    'run_output_dir': str(single_run_config['run_dir']),
    'planned_run_count': len(planned_sweep_runs),
    'inferred_execution': EFFECTIVE_EXECUTION_MODE,
}
print('Resolved first-run configuration summary:')
print(json.dumps(resolved_configuration_summary, indent=2, sort_keys=True))
print()
print_run_checks(single_run_config, label='First planned grid run')
print('\nFirst planned grid command:')
print(command_string(train_cmd))
print('First planned grid output directory:', target_run_dir)
print('\nTotal planned grid runs:', len(planned_sweep_runs))
print('Inferred execution:', EFFECTIVE_EXECUTION_MODE)
print('Sweep status path:', sweep_status_path)
print('Sweep comparison CSV:', final_comparison_csv)
print('Sweep output directory:', local_runs_dir)
if pd is not None:
    display(pd.DataFrame(planned_sweep_table))
else:
    for row in planned_sweep_table:
        print(row)


## 9. Run Training

This is the only section that starts training. The notebook infers the behavior from the planned grid: one planned configuration runs once; more than one planned configuration runs as a sweep.


In [ ]:
#@title Execute training
EFFECTIVE_EXECUTION_MODE = infer_execution_mode(len(planned_sweep_runs))
RUN_SINGLE_MODE = EFFECTIVE_EXECUTION_MODE == 'single_run'
RUN_SWEEP_MODE = EFFECTIVE_EXECUTION_MODE == 'sweep'

print('Planned grid runs:', len(planned_sweep_runs))
print('Inferred execution:', EFFECTIVE_EXECUTION_MODE)
if (RUN_SINGLE_MODE or RUN_SWEEP_MODE) and DEVICE == 'cuda':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('DEVICE is cuda, but torch.cuda.is_available() is False. Change DEVICE to cpu or enable a GPU runtime.')
if RUN_SINGLE_MODE:
    validate_run_before_training(single_run_config)
    print('=' * 60)
    print('Starting first planned grid run')
    print('Run name:', single_run_config['run_name'])
    print('Task:', single_run_config['task'])
    print('Model:', single_run_config['model_preset'])
    print('RING edge mode:', single_run_config['ring_edges_mode'])
    print('Uses ESM:', single_run_config['uses_esm'])
    print('Learning rate:', single_run_config['learning_rate'])
    print('GVP/capacity:', {key: single_run_config[key] for key in ['gvp_layers', 'hidden_s', 'hidden_v', 'edge_hidden', 'edge_radius', 'esm_fusion_dim', 'head_mlp_layers']})
    print('Selection metric:', single_run_config['selection_metric'])
    print('Output directory:', single_run_config['run_dir'])
    print('=' * 60)
    return_code = stream_command(single_run_config['command'], cwd=repo_dir, env=env_for_run(single_run_config))
    if return_code != 0:
        failed_run_dirs.append(single_run_config['run_dir'])
        print(f'Single run failed with exit code {return_code}: {single_run_config["run_dir"]}')
    else:
        completed_run_dirs.append(single_run_config['run_dir'])
        summary = summarize_completed_run(single_run_config['run_dir'])
        print('Single run succeeded:', single_run_config['run_dir'])
        print('Selected checkpoint:', summary.get('selected_checkpoint'))
        print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
        print('Held-out test metric summary:', summary.get('test_summary'))

if RUN_SWEEP_MODE:
    sweep_status_rows = []
    total_runs = len(planned_sweep_runs)
    for index, run in enumerate(planned_sweep_runs, start=1):
        sweep_status_rows.append(status_row_for_run(index, total_runs, run))
    write_sweep_status(sweep_status_rows, sweep_status_path)

    print('Total planned runs:', total_runs)
    print('Sweep status CSV:', sweep_status_path)
    print('Sweep outputs directory:', local_runs_dir)
    if pd is not None:
        display(pd.DataFrame(planned_table_rows(planned_sweep_runs)))
    else:
        for row in planned_table_rows(planned_sweep_runs):
            print(row)

    for index, run in enumerate(planned_sweep_runs, start=1):
        validate_run_before_training(run)
        status_row = sweep_status_rows[index - 1]
        if SKIP_EXISTING_RUNS and run['run_dir'].exists():
            summary = summarize_completed_run(run['run_dir'])
            mark_status(status_row, 'skipped')
            status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            status_row['error_message'] = 'run directory already exists'
            write_sweep_status(sweep_status_rows, sweep_status_path)
            completed_run_dirs.append(run['run_dir'])
            print(f'Skipping existing run directory: {run["run_dir"]}')
            continue

        print('=' * 60)
        print(f'Starting run {index} / {total_runs}')
        print('Run name:', run['run_name'])
        print('Task:', run['task'])
        print('Model:', run['model_preset'])
        print('RING edge mode:', run['ring_edges_mode'])
        print('Uses ESM:', run['uses_esm'])
        print('Learning rate:', run['learning_rate'])
        print('GVP/capacity:', {key: run[key] for key in ['gvp_layers', 'hidden_s', 'hidden_v', 'edge_hidden', 'edge_radius', 'esm_fusion_dim', 'head_mlp_layers']})
        print('Selection metric:', run['selection_metric'])
        print('Output directory:', run['run_dir'])
        print('=' * 60)

        mark_status(status_row, 'running')
        status_row['start_time'] = datetime.now().isoformat(timespec='seconds')
        write_sweep_status(sweep_status_rows, sweep_status_path)
        return_code = stream_command(run['command'], cwd=repo_dir, env=env_for_run(run))
        status_row['end_time'] = datetime.now().isoformat(timespec='seconds')
        if return_code == 0:
            mark_status(status_row, 'succeeded')
            completed_run_dirs.append(run['run_dir'])
            summary = summarize_completed_run(run['run_dir'])
            status_row['selected_checkpoint'] = summary.get('selected_checkpoint') or ''
            status_row['selected_metric_value'] = '' if summary.get('selected_metric_value') is None else summary.get('selected_metric_value')
            status_row['test_summary'] = json.dumps(summary.get('test_summary') or {}, sort_keys=True)
            print(f'Run {index} succeeded:', run['run_dir'])
            print('Selected checkpoint:', summary.get('selected_checkpoint'))
            print('Best validation metric:', summary.get('selection_metric'), summary.get('selected_metric_value'))
            print('Held-out test metric summary:', summary.get('test_summary'))
        else:
            mark_status(status_row, 'failed')
            status_row['error_message'] = f'exit code {return_code}'
            status_row['test_summary'] = ''
            failed_run_dirs.append(run['run_dir'])
            print(f'Run {index} failed with exit code {return_code}: {run["run_dir"]}')
            if STOP_ON_FIRST_FAILURE:
                write_sweep_status(sweep_status_rows, sweep_status_path)
                raise RuntimeError(f'Stopping sweep after failed run {index}: {run["run_name"]}')
        write_sweep_status(sweep_status_rows, sweep_status_path)

    report_code = run_report(local_runs_dir, final_comparison_csv, final_comparison_figure)
    if report_code != 0:
        print(f'Warning: report_runs.py failed with exit code {report_code}')
    succeeded = sum(1 for row in sweep_status_rows if row['status'] == 'succeeded')
    skipped = sum(1 for row in sweep_status_rows if row['status'] == 'skipped')
    failed = sum(1 for row in sweep_status_rows if row['status'] == 'failed')
    print('=' * 60)
    print('Sweep finished')
    print('Succeeded:', succeeded)
    print('Skipped:', skipped)
    print('Failed:', failed)
    print('RING edge modes:', sorted({run['ring_edges_mode'] for run in planned_sweep_runs}))
    print('ESM-enabled planned runs:', sum(1 for run in planned_sweep_runs if run['uses_esm']))
    print('Sweep status CSV:', sweep_status_path)
    print('Final comparison CSV:', final_comparison_csv)
    if RUN_MODE == 'colab':
        print('Copied Drive outputs after running the copy cell:', drive_output_dir if globals().get('use_drive_paths', False) else '(Drive paths disabled)')
    else:
        print('Local outputs directory:', OUTPUT_DIR)
    print('=' * 60)


## 10. Copy Run Outputs Back to Drive

This section is Colab-only. In local mode, outputs already remain under `OUTPUT_DIR`.


In [ ]:
#@title Copy outputs to Drive
COPY_OUTPUTS_TO_DRIVE = RUN_MODE == "colab" and globals().get('use_drive_paths', False)  #@param {type:"boolean"}

import shutil
from pathlib import Path

if 'completed_run_dirs' not in globals():
    completed_run_dirs = []
if 'failed_run_dirs' not in globals():
    failed_run_dirs = []

drive_runs_dir = drive_output_dir / 'runs'
drive_sweep_dir = drive_output_dir / 'sweeps'
run_dirs_to_copy = []
for candidate in completed_run_dirs + failed_run_dirs:
    candidate = Path(candidate)
    if candidate.exists() and candidate not in run_dirs_to_copy:
        run_dirs_to_copy.append(candidate)
if not run_dirs_to_copy and 'target_run_dir' in globals() and target_run_dir.exists():
    run_dirs_to_copy.append(target_run_dir)

if RUN_MODE == "colab" and COPY_OUTPUTS_TO_DRIVE and globals().get('use_drive_paths', False):
    drive_runs_dir.mkdir(parents=True, exist_ok=True)
    copied = []
    for run_dir in run_dirs_to_copy:
        drive_run_dir = drive_runs_dir / run_dir.name
        if drive_run_dir.exists():
            shutil.rmtree(drive_run_dir)
        shutil.copytree(run_dir, drive_run_dir)
        copied.append(drive_run_dir)
        print('Copied run output to:', drive_run_dir)

    drive_sweep_dir.mkdir(parents=True, exist_ok=True)
    for artifact in [sweep_status_path, final_comparison_csv, final_comparison_figure]:
        artifact = Path(artifact)
        if artifact.exists():
            destination = drive_sweep_dir / artifact.name
            shutil.copy2(artifact, destination)
            print('Copied sweep artifact to:', destination)
    if not copied:
        print('No completed run directories found to copy yet.')
    print('Drive output directory:', drive_output_dir)
elif RUN_MODE == "colab" and not globals().get('use_drive_paths', False):
    print('Colab runtime-local mode: Drive copy skipped. Outputs remain under:', OUTPUT_DIR)
elif RUN_MODE == "local":
    print('Local mode: Drive copy skipped. Outputs remain under:', OUTPUT_DIR)
else:
    print('Copy skipped.')



## 11. Summarize Runs with `src/report_runs.py`

The summary CSV is the main comparison table. The optional figure is created only when matplotlib is available and the selected validation metric is numeric. Sweep execution already runs this once at the end; this section lets you regenerate the report from local or copied Drive outputs.

In [ ]:
#@title Generate run summary
SUMMARY_SOURCE = 'local_runs' if RUN_MODE == "local" else 'drive_outputs'  #@param ['local_runs', 'drive_outputs']
SUMMARY_BASENAME = 'baseline_first_summary'  #@param {type:"string"}

if 'drive_runs_dir' not in globals():
    drive_runs_dir = drive_output_dir / 'runs'
    print('Warning: drive_runs_dir was not set by the copy cell; defaulting to', drive_runs_dir)

summary_runs_dir = local_runs_dir if SUMMARY_SOURCE == 'local_runs' else drive_runs_dir
summary_output_dir = OUTPUT_DIR if RUN_MODE == "local" else drive_output_dir
summary_csv = summary_output_dir / f'{SUMMARY_BASENAME}.csv'
summary_figure = summary_output_dir / f'{SUMMARY_BASENAME}.png'
summary_output_dir.mkdir(parents=True, exist_ok=True)

if not summary_runs_dir.exists():
    print(f'No run directory found yet: {summary_runs_dir}')
else:
    report_code = run_report(summary_runs_dir, summary_csv, summary_figure)
    if report_code != 0:
        raise RuntimeError(f'report_runs.py failed with exit code {report_code}')
    print('Summary CSV:', summary_csv)
    print('Summary figure:', summary_figure)



## 12. Display the Summary Table and Figure

Use this table to compare runs by validation-selected metrics first. Test metrics are for final held-out reporting after selecting checkpoints by validation performance.

In [ ]:
from IPython.display import Image, display
import pandas as pd

if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    display(summary_df)
else:
    print(f'Summary CSV not found yet: {summary_csv}')

if summary_figure.exists():
    display(Image(filename=str(summary_figure)))
else:
    print(f'Optional summary figure not found: {summary_figure}')


## 13. Final Model-Selection Checklist

- Confirm every final run used the non-overlapped PinMyMetal split, or clearly label exact/possibly-overlapped runs as secondary reference only.
- Compare Only-GVP, Only-ESM, GVP + late fusion, and GVP + early fusion before moving to complex fusion modes.
- Select checkpoints and model variants by validation metrics, not by repeatedly checking the held-out test set.
- For metal prediction, report both 6-class metrics and collapsed-4 metrics from the selected checkpoint.
- For EC prediction, report supported EC level metrics from the selected checkpoint.
- Preserve `run_config.json`, `run_metadata.json`, `dataset_summary.json`, `test_report.json`, selected checkpoint path, split identity, overlap status, and the summary CSV/figure in Drive.